# Column Prepare — explore แล้วแก้ทีละเรื่อง (interleaved)

ทำ prepare ครบทั้งหมด (text-clean + value-format + property-decompose) เขียนออก `processed/prepared/*.csv` โดยตรง — `eda_prepare.ipynb` เหลือแค่ EDA อย่างเดียว อ่านต่อจากที่นี่

แต่ละปัญหาเรียง **explore ก่อน แล้ว fix ตามทันที** แบ่ง 3 ช่วงใหญ่:
- **ช่วง 1 — Clean dirty text** (A/B/C): unicode/whitespace, typo, spelling variant
- **ช่วง 2 — Clean value formats** (D–I): thickness, stability, purity, material family, unit ผิด
- **ช่วง 3 — Prepare properties**: แตกชื่อ property, แปลงสเกล FF

ปิดท้ายด้วยหัวข้อที่**ยัง explore อย่างเดียว ยังไม่มี fix จริง** (process_type canon, atmosphere, unit-audit ที่เหลือ, cross-table)

## Setup

In [1]:
import os, re, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "normalize").exists() else Path.cwd().parent
os.chdir(ROOT)
SRC = ROOT / "processed" / "normalize"
AUDIT = ROOT / "processed" / "audit"

# dtype=str -> pandas ไม่แปลงตัวเลข (100 -> 100.0) เขียนกลับได้ตรงต้นฉบับ นี่คือตารางที่จะถูกแก้จริง
tables = {p.stem: pd.read_csv(p, dtype=str) for p in sorted(SRC.glob("*.csv"))}

# FIX: materials/process มี NaN ใน LayerID ตอน ETL -> pandas cast เป็น float -> "1.0" ขณะที่ layers เป็น "1"
# ทำให้ merge on layer_id ได้ 0 แถว (join พังเงียบ ๆ) -> normalize ให้เป็นสตริงจำนวนเต็มทุกตารางตั้งแต่โหลด
for _name, _df in tables.items():
    if "layer_id" in _df.columns:
        _lid = pd.to_numeric(_df["layer_id"], errors="coerce").astype("Int64")
        _df["layer_id"] = _lid.astype(str).replace("<NA>", pd.NA)
tables_raw = {name: df.copy() for name, df in tables.items()}   # สำเนา "ก่อน clean" ไว้เทียบตอนจบ

diff_log = []   # ทุกเซลล์ที่ถูกแก้ (PART text-clean): step, table, column, row, device_id, before, after

def log_and_apply(name: str, col: str, new: pd.Series, step: str) -> int:
    """เทียบคอลัมน์เดิมกับค่าใหม่ บันทึกเซลล์ที่ต่างลง diff_log แล้วค่อย apply"""
    df = tables[name]
    old = df[col]
    mask = old.notna() & (old != new)
    for idx in df.index[mask]:
        diff_log.append({"step": step, "table": name, "column": col, "row": int(idx),
                         "device_id": df.at[idx, "device_id"] if "device_id" in df.columns else "",
                         "before": old.at[idx], "after": new.at[idx]})
    df[col] = new.astype(object)
    return int(mask.sum())

def step_diff(step: str) -> pd.DataFrame:
    d = pd.DataFrame([r for r in diff_log if r["step"] == step],
                     columns=["step", "table", "column", "row", "device_id", "before", "after"])
    return (d.groupby(["table", "column", "before", "after"])
              .size().rename("n_cells").reset_index())

print("โหลด", len(tables), "ตาราง:", {k: v.shape for k, v in tables.items()})

โหลด 7 ตาราง: {'devices': (95, 4), 'layers': (760, 5), 'materials': (1205, 7), 'process': (1417, 9), 'properties': (1118, 6), 'stability': (90, 10), 'vendors': (1551, 4)}


### Master column inventory (ภาพรวมก่อนเริ่ม)

ตารางเดียวเห็นทุกคอลัมน์: ชนิด, %missing, จำนวนค่าไม่ซ้ำ, จำนวนกลุ่มที่สะกดต่าง (text), และอัตราที่ parse เป็นตัวเลขได้ (quant)

In [2]:
QUANT_COLS = {"thickness_nm","amount","parameter","value","error","sample_no",
              "temperature_c","humidity_pct","time_hr","retention_pct","t80_hr",
              "initial_pce","final_pce"}
SKIP_COLS = {"device_id","source_file","source_sheet","layer_id","step_order","doi_raw"}

def kind(col):
    return "quant" if col in QUANT_COLS else "text"

rows = []
for name, df in tables.items():
    for col in df.columns:
        if col in SKIP_COLS: continue
        s = df[col]
        info = {"table": name, "column": col, "kind": kind(col),
                "n_nonnull": int(s.notna().sum()),
                "n_unique": int(s.dropna().astype(str).str.strip().nunique()),
                "n_missing": int(s.isna().sum())}
        rows.append(info)
inventory = pd.DataFrame(rows)
inventory

,table,column,kind,n_nonnull,n_unique,n_missing
0,devices,sample_no,quant,95,5,0
1,layers,layer_type,text,757,13,3
2,layers,material,text,658,124,102
3,layers,thickness_nm,quant,249,46,511
4,materials,material,text,1113,107,92
5,materials,type,text,850,36,355
6,materials,solvent,text,756,56,449
7,materials,amount,quant,691,116,514
8,materials,unit,text,664,36,541
9,process,process_type,text,1338,40,79


# ช่วง 1 — Clean dirty text

## A. Unicode ปน / whitespace เกิน — explore

In [3]:
bad_chars = sum(int(tables[n][c].fillna("").str.contains((chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"])).sum())
                for n in tables for c in tables[n].columns)
print(f"เจอ {bad_chars} เซลล์ ที่มี Greek mu (μ, U+03BC) / unicode minus (−, U+2212) / NBSP (U+00A0) ปนอยู่")
print("ตัวอย่างที่พบ (ถ้ามี):")
for n in tables:
    for c in tables[n].columns:
        s = tables[n][c].fillna("")
        hit = s[s.str.contains((chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"]))]
        if len(hit):
            print(f"  {n}.{c}: {list(hit.unique()[:10])}")

เจอ 64 เซลล์ ที่มี Greek mu (μ, U+03BC) / unicode minus (−, U+2212) / NBSP (U+00A0) ปนอยู่
ตัวอย่างที่พบ (ถ้ามี):
  materials.unit: ['μL']
  process.unit: ['μL, sec', 'μL', 'μm, mm/sec']
  properties.unit: ['mA cm−2', 'μm']


## A. Unicode ปน / whitespace เกิน — fix

In [4]:
UNICODE_FIX = str.maketrans({chr(0x03BC): chr(0x00B5),   # Greek mu -> micro sign
                             chr(0x2212): "-",              # Unicode minus -> hyphen
                             chr(0x00A0): " "})             # NBSP -> space

def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.translate(UNICODE_FIX)).strip()

n_fixed = 0
for name, df in tables.items():
    for col in df.columns:
        after = df[col].map(lambda v: norm_text(v) if isinstance(v, str) else v)
        n_fixed += log_and_apply(name, col, after, "1-unicode/ws")
print("1-unicode/ws: แก้", n_fixed, "เซลล์")
print("before -> after ไม่ซ้ำ:")
step_diff("1-unicode/ws")

1-unicode/ws: แก้ 64 เซลล์
before -> after ไม่ซ้ำ:


,table,column,before,after,n_cells
0,materials,unit,μL,µL,2
1,process,unit,μL,µL,7
2,process,unit,"μL, sec","µL, sec",31
3,process,unit,"μm, mm/sec","µm, mm/sec",19
4,properties,unit,mA cm−2,mA cm-2,4
5,properties,unit,μm,µm,1


## B. Typo ที่รู้ตัวแล้ว — explore

In [5]:
TYPO_FIX = {
    ("properties", "property"): {
        "Trap densitiy": "Trap density",  # สะกดผิด
        "AverageVoc": "Average Voc",      # space หาย
    },
}
for (name, col), fixes in TYPO_FIX.items():
    for typo, correct in fixes.items():
        n = int((tables[name][col] == typo).sum())
        print(f"{name}.{col}: {typo!r} ผิด {n} แถว (ควรเป็น {correct!r})")

properties.property: 'Trap densitiy' ผิด 0 แถว (ควรเป็น 'Trap density')
properties.property: 'AverageVoc' ผิด 1 แถว (ควรเป็น 'Average Voc')


## B. Typo — fix

In [6]:
n_fixed = 0
for (name, col), fixes in TYPO_FIX.items():
    n_fixed += log_and_apply(name, col, tables[name][col].replace(fixes), "2-typo")
print("2-typo: แก้", n_fixed, "เซลล์")
print("before -> after ไม่ซ้ำ:")
step_diff("2-typo")

2-typo: แก้ 1 เซลล์
before -> after ไม่ซ้ำ:


,table,column,before,after,n_cells
0,properties,property,AverageVoc,Average Voc,1


## C. สะกดต่างกัน (case/วรรคตอน) — explore

In [7]:
TEXT_COLS = {
    "layers":     ["layer_type", "material"],
    "materials":  ["material", "type", "solvent", "unit"],
    "process":    ["process_type", "description", "unit"],
    "properties": ["property", "unit"],
    "stability":  ["environment", "encapsulation"],
    "vendors":    ["material", "vendor"],
}

PREFERRED = {
    ("process", "process_type", "spincoating"): "Spin coating",
    ("process", "process_type", "thermalevaporation"): "Thermal evaporation",
    ("materials", "solvent", "dmfdmso41vv"): "DMF/DMSO (4:1 v/v)",
    ("materials", "solvent", "dmfdmso3081vv"): "DMF/DMSO (3.08/1 v/v)",
    ("materials", "solvent", "dmfdmso41"): "DMF/DMSO (4:1)",
    ("properties", "property", "grainsize"): "Grain size",  # เสมอกัน 10:10
}

def variant_key(s: str) -> str:
    return re.sub(r"[\W_]", "", s.lower())

map_rows = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        counts = Counter(tables[name][col].dropna())
        groups = {}
        for val, n in counts.items():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):
                groups.setdefault(k, []).append((val, n))
        for k, variants in groups.items():
            if len(variants) < 2 and (name, col, k) not in PREFERRED:
                continue
            canonical = PREFERRED.get((name, col, k)) or \
                        max(variants, key=lambda v: (v[1], v[0]))[0]
            for val, n in variants:
                if val != canonical:
                    map_rows.append({"table": name, "column": col,
                                     "raw": val, "canonical": canonical, "n_rows": n,
                                     "rule": "override" if (name, col, k) in PREFERRED
                                             else "majority"})

var_map = (pd.DataFrame(map_rows,
                        columns=["table", "column", "raw", "canonical", "n_rows", "rule"])
             .sort_values(["table", "column", "canonical", "raw"])
             .reset_index(drop=True))
print(f"พบ {len(var_map)} แถว mapping ({var_map[['table','column','canonical']].drop_duplicates().shape[0]} กลุ่ม)")
var_map

พบ 66 แถว mapping (54 กลุ่ม)


,table,column,raw,canonical,n_rows,rule
0,layers,layer_type,HTL Treatment,HTL treatment,1,majority
1,layers,layer_type,PSK Treatment,PSK treatment,1,majority
2,layers,material,Cs/FA/MA,CsFAMA,1,majority
3,layers,material,Cs/FA/MA triple-cation perovskite,CsFAMA triple-cation perovskite,1,majority
4,layers,material,NiOx (Sputtered),NiOx (sputtered),1,majority
5,layers,material,SnOx (,SnOx,1,majority
6,materials,solvent,DMF /DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
7,materials,solvent,DMF/DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
8,materials,solvent,DMFDMSO (3.08/1 v/v),DMF/DMSO (3.08/1 v/v),2,override
9,materials,solvent,DMF/DMSO (4/1 V/V),DMF/DMSO (4:1 v/v),36,override


## C. สะกดต่างกัน — fix

In [8]:
report = []
for (name, col), g in var_map.groupby(["table", "column"]):
    mapping = dict(zip(g["raw"], g["canonical"]))
    before_unique = tables[name][col].nunique()
    n = log_and_apply(name, col, tables[name][col].replace(mapping), "3-variant")
    after_unique = tables[name][col].nunique()
    report.append({"table": name, "column": col, "n_cells": n,
                   "n_unique_before": before_unique, "n_unique_after": after_unique})
print("สรุปต่อคอลัมน์:")
display(pd.DataFrame(report))
print("before -> after ไม่ซ้ำ (ทุกคู่ raw -> canonical):")
step_diff("3-variant")

สรุปต่อคอลัมน์:


,table,column,n_cells,n_unique_before,n_unique_after
0,layers,layer_type,2,13,11
1,layers,material,4,124,120
2,materials,solvent,266,56,43
3,materials,type,4,36,35
4,materials,unit,26,35,32
5,process,description,3,76,73
6,process,process_type,621,40,32
7,process,unit,26,43,37
8,properties,property,50,116,107
9,properties,unit,19,25,21


before -> after ไม่ซ้ำ (ทุกคู่ raw -> canonical):


,table,column,before,after,n_cells
0,layers,layer_type,HTL Treatment,HTL treatment,1
1,layers,layer_type,PSK Treatment,PSK treatment,1
2,layers,material,Cs/FA/MA,CsFAMA,1
3,layers,material,Cs/FA/MA triple-cation perovskite,CsFAMA triple-cation perovskite,1
4,layers,material,NiOx (Sputtered),NiOx (sputtered),1
5,layers,material,SnOx (,SnOx,1
6,materials,solvent,DMF /DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1
7,materials,solvent,DMF/DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1
8,materials,solvent,DMF/DMSO (4/1 V/V),DMF/DMSO (4:1 v/v),36
9,materials,solvent,DMF/DMSO (4:1 V/V),DMF/DMSO (4:1 v/v),58


## สรุป + save audit + verify (จบช่วง 1)

In [9]:
diff_cells = pd.DataFrame(diff_log,
                          columns=["step", "table", "column", "row", "device_id",
                                   "before", "after"])
recount = sum(int((tables_raw[n][c].notna() & (tables_raw[n][c] != tables[n][c])).sum())
              for n in tables for c in tables[n].columns)
n_cells_logged = len(diff_cells[["table", "column", "row"]].drop_duplicates())
ok = "ตรงกัน ✓" if recount == n_cells_logged else "ไม่ตรง — ต้องตรวจ!"
print(f"เซลล์ที่แก้ (นับจาก log)      : {n_cells_logged}")
print(f"เซลล์ที่แก้ (นับอิสระจาก raw) : {recount}  ({ok})")

เซลล์ที่แก้ (นับจาก log)      : 1201
เซลล์ที่แก้ (นับอิสระจาก raw) : 1201  (ตรงกัน ✓)


In [10]:
AUDIT.mkdir(parents=True, exist_ok=True)
var_map.to_csv(AUDIT / "text_cleanup_map.csv", index=False)
diff_cells.to_csv(AUDIT / "clean_diff_cells.csv", index=False)
print("wrote", AUDIT / "text_cleanup_map.csv")
print("wrote", AUDIT / "clean_diff_cells.csv")

wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\text_cleanup_map.csv
wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\clean_diff_cells.csv


# ช่วง 2 — Clean value formats

## D. ภาพรวมคอลัมน์ตัวเลขทั้งหมด — explore

In [11]:
tables_after_text = {name: df.copy() for name, df in tables.items()}   # snapshot ไว้ verify ท้ายช่วง 2

CONFIG = {
    "RT_CELSIUS": 25.0,
    "ANGSTROM_PER_NM": 10.0,
    "RANGE_REPRESENTATIVE": "mid",
}
NUM = r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?"

def is_num(s) -> bool:
    return isinstance(s, str) and re.fullmatch(NUM, s.strip()) is not None

# ค่าที่เขียนแบบ "1.01×10-3" / "1.01 x 10^-3" -- pandas อ่านไม่ออก ต้องแปลงเป็น 1.01e-3 ก่อน
#   (มากับ Raw Data ชุดที่ 2: Conductivity ของ solener.2024.112888)
SCI_RE = re.compile(rf"({NUM})\s*[×xX·]\s*10\s*\^?\s*([+-]?\d+)")

def to_num_sci(v):
    """แปลงเป็น float โดยรองรับสัญกรณ์ยกกำลังแบบเขียนมือ -- คืน (ค่า, note)"""
    if not isinstance(v, str):
        return pd.to_numeric(v, errors="coerce"), ""
    s = v.strip()
    m = SCI_RE.fullmatch(s)
    if m:
        x = float(m[1]) * 10 ** int(m[2])
        return x, f"สัญกรณ์ยกกำลัง: {s} -> {x:g}"
    return pd.to_numeric(s, errors="coerce"), ""

audit_rows = []  # ทุกแถวที่ถูกตีความจะถูกบันทึกลงนี่

NUM_COLS = {
    "layers":     ["thickness_nm"],
    "materials":  ["amount"],
    "properties": ["value", "error"],
    "stability":  ["temperature_c", "humidity_pct", "time_hr", "retention_pct",
                   "t80_hr", "initial_pce", "final_pce"],
    "vendors":    ["purity"],
}
scan = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].dropna().str.strip()
        s = s[s != ""]
        bad = s[~s.map(is_num)]
        scan.append({"table": name, "column": col,
                     "n_filled": len(s), "n_bad": len(bad),
                     "bad_examples": ", ".join(sorted(bad.unique())[:5])})
scan = pd.DataFrame(scan).sort_values("n_bad", ascending=False).reset_index(drop=True)
print(f"แถวที่ parse ไม่ได้รวม: {scan.n_bad.sum()}")
scan

แถวที่ parse ไม่ได้รวม: 135


,table,column,n_filled,n_bad,bad_examples
0,vendors,purity,664,69,"20 wt, 80-90, 9 - 15 Ω □-1, 99.9%, > 99"
1,stability,humidity_pct,42,29,"10 ± 5, 25 ± 5, 30 - 50, 30 ± 5, 30-40"
2,layers,thickness_nm,249,20,"1/25, 1200 Å, 280-320, 4, 100, 40-60"
3,stability,temperature_c,66,10,"20 ± 5, 50 ± 5, RT"
4,materials,amount,691,2,1:50
5,properties,value,1092,2,"1.01×10-3, 1.21×10-3"
6,stability,retention_pct,74,2,"< 20, < 55"
7,stability,t80_hr,2,1,> 1300
8,properties,error,202,0,
9,stability,time_hr,83,0,


## E. `layers.thickness_nm` — explore

In [12]:
s = tables["layers"]["thickness_nm"].dropna().str.strip()
bad = s[(s != "") & ~s.map(is_num)]
print(f"{len(bad)} แถว parse ตรงๆ ไม่ได้ (หน่วย Å ปน / ช่วงค่า / หลายชั้นย่อยในเซลล์เดียว):")
tables["layers"].loc[bad.index, ["device_id", "layer_type", "material", "thickness_nm"]]

20 แถว parse ตรงๆ ไม่ได้ (หน่วย Å ปน / ช่วงค่า / หลายชั้นย่อยในเซลล์เดียว):


,device_id,layer_type,material,thickness_nm
3,10.1002admi.202500231_Sample1,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
11,10.1002admi.202500231_Sample2,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
19,10.1002admi.202500231_Sample3,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
27,10.1002admi.202500231_Sample4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
35,10.1007_s10854-017-8094-9_Sample1,PSK,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,280-320
59,10.1007_s10854-017-8094-9_Sample4,PSK,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,280-320
149,10.1016_j.solmat.2025.113858_Sample1,ETL,LiF/C60,1/25
157,10.1016_j.solmat.2025.113858_Sample2,ETL,LiF/C60,1/25
165,10.1016_j.solmat.2025.113858_Sample3,ETL,LiF/C60,1/25
173,10.1016_j.solmat.2025.113858_Sample4,ETL,LiF/C60,1/25


In [13]:
s = tables["layers"]["thickness_nm"].dropna().str.strip()
bad = s[(s != "") & ~s.map(is_num)]

def classify_bad_thickness(v):
    v = v.strip()
    if "Å" in v:
        return "unit_angstrom (ค่าเดียว หน่วย Å, ต้อง x0.1 -> nm)"
    if re.search(r"\d\s*-\s*\d", v):
        return "range (เช่น 400-450)"
    if "," in v:
        return "multi_value_comma (หลายชั้นย่อยในเซลล์เดียว)"
    return "other"

counts = bad.map(classify_bad_thickness).value_counts()
print(counts)
print()
for cat in counts.index:
    ex = bad[bad.map(classify_bad_thickness) == cat].unique()[:5]
    print(f"  {cat}: {list(ex)}")

thickness_nm
range (เช่น 400-450)                                 8
unit_angstrom (ค่าเดียว หน่วย Å, ต้อง x0.1 -> nm)    6
other                                                4
multi_value_comma (หลายชั้นย่อยในเซลล์เดียว)         2
Name: count, dtype: int64

  range (เช่น 400-450): ['400-450', '280-320', '40-60']
  unit_angstrom (ค่าเดียว หน่วย Å, ต้อง x0.1 -> nm): ['400 Å', '90 Å', '1200 Å']
  other: ['1/25']
  multi_value_comma (หลายชั้นย่อยในเซลล์เดียว): ['4, 100']


## E. `layers.thickness_nm`, `material` - fix

In [14]:
def parse_thickness(v):
    """คืน (parsed, vmin, vmax, note) — note ว่าง = ตัวเลขปกติไม่ต้องตีความ"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):                                     # ตัวเลขปกติ
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*Å", s)              # หน่วย Å
    if m:
        x = float(m[1]) / CONFIG["ANGSTROM_PER_NM"]
        return (x, x, x, f"แปลงหน่วย: {s} = {x:g} nm (1 Å = 0.1 nm)")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)  # ช่วง a-b
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    # ชั้นซ้อนที่เขียนด้วย "/" ให้ตรงกับชื่อวัสดุ เช่น material="LiF/C60" thickness="1/25"
    #   (มากับ Raw Data ชุดที่ 2 -- ความหมายเดียวกับ "a, b" ที่รองรับอยู่แล้ว)
    m = re.fullmatch(rf"({NUM})\s*/\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return (a + b, min(a, b), max(a, b),
                f"ชั้นซ้อนคั่นด้วย /: ผลรวม {a + b:g} nm (ชั้นย่อย {a:g} + {b:g})")
    m = re.fullmatch(rf"({NUM})\s*,\s*({NUM})", s)     # หลายชั้นย่อย a, b
    if m:
        a, b = float(m[1]), float(m[2])
        return (a + b, min(a, b), max(a, b),
                f"หลายชั้นย่อยในเซลล์เดียว: ผลรวม {a + b:g} nm (ชั้นย่อย {a:g} + {b:g})")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

lay = tables["layers"]
parsed = lay["thickness_nm"].map(parse_thickness)
lay["thickness_nm_parsed"] = parsed.map(lambda t: t[0])
lay["thickness_nm_min"] = parsed.map(lambda t: t[1])
lay["thickness_nm_max"] = parsed.map(lambda t: t[2])
lay["thickness_nm_note"] = parsed.map(lambda t: t[3])

hit = lay["thickness_nm_note"] != ""
for _, r in lay[hit].iterrows():
    audit_rows.append({"table": "layers", "column": "thickness_nm",
                       "device_id": r["device_id"], "raw": r["thickness_nm"],
                       "parsed": r["thickness_nm_parsed"], "vmin": r["thickness_nm_min"],
                       "vmax": r["thickness_nm_max"], "extra": r["layer_type"],
                       "note": r["thickness_nm_note"]})

print(f"แถวที่ถูกตีความ: {int(hit.sum())}")
lay.loc[hit, ["device_id", "layer_type", "material", "thickness_nm",
              "thickness_nm_parsed", "thickness_nm_min", "thickness_nm_max",
              "thickness_nm_note"]].rename(
    columns={"thickness_nm": "before (raw)", "thickness_nm_parsed": "after (parsed)"})

แถวที่ถูกตีความ: 20


,device_id,layer_type,material,before (raw),after (parsed),thickness_nm_min,thickness_nm_max,thickness_nm_note
3,10.1002admi.202500231_Sample1,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
11,10.1002admi.202500231_Sample2,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
19,10.1002admi.202500231_Sample3,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
27,10.1002admi.202500231_Sample4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
35,10.1007_s10854-017-8094-9_Sample1,PSK,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,280-320,300.0,280.0,320.0,"ช่วงค่า: ใช้ค่ากลาง 300 (min=280, max=320)"
59,10.1007_s10854-017-8094-9_Sample4,PSK,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,280-320,300.0,280.0,320.0,"ช่วงค่า: ใช้ค่ากลาง 300 (min=280, max=320)"
149,10.1016_j.solmat.2025.113858_Sample1,ETL,LiF/C60,1/25,26.0,1.0,25.0,ชั้นซ้อนคั่นด้วย /: ผลรวม 26 nm (ชั้นย่อย 1 + 25)
157,10.1016_j.solmat.2025.113858_Sample2,ETL,LiF/C60,1/25,26.0,1.0,25.0,ชั้นซ้อนคั่นด้วย /: ผลรวม 26 nm (ชั้นย่อย 1 + 25)
165,10.1016_j.solmat.2025.113858_Sample3,ETL,LiF/C60,1/25,26.0,1.0,25.0,ชั้นซ้อนคั่นด้วย /: ผลรวม 26 nm (ชั้นย่อย 1 + 25)
173,10.1016_j.solmat.2025.113858_Sample4,ETL,LiF/C60,1/25,26.0,1.0,25.0,ชั้นซ้อนคั่นด้วย /: ผลรวม 26 nm (ชั้นย่อย 1 + 25)


### Pivot Cleaning `material`

**1. สเต็ป Pivot**
* แตก Dict สัดส่วนด้วย `json_normalize` -> Pivot (1 แถว = 1 Device) -> จะเกิด `NaN` ตามธรรมชาติ

**2. การรับมือ NaN**
*  **Non-Perovskite (เช่น ITO, Ag):** ปล่อย `NaN` ไว้ (ห้ามเติม 0 เด็ดขาด เพราะ 0 หมายถึงมีธาตุนั้น 0%)
*  **สารเดี่ยว (Plain):** คอลัมน์สารผสม (`mat_2`, `blend_ratio`) ปล่อย `NaN` ได้เลย ถือว่าปกติ
*  **Perovskite สูตรย่อ (เช่น CsFAMA):** มีธาตุ Pb/Sn แต่สัดส่วนเป็น `NaN` -> **ห้ามปล่อยผ่าน ต้อง Flag แยกไปแก้ Manual**

> ⚠️ **เซลล์ชุดนี้เป็น prototype ที่ไม่ได้ใช้ต่อ** — `layers_material_parsed`/`mat_df` ไม่เคยถูก merge กลับหรือ export ตัวจริงที่แตก composition คือ **§J** ด้านล่าง (ผล export อยู่ที่ `prepared_v2/psk_composition.csv` ผ่าน recheck §8.5)

In [15]:
def classify_material_format(v):
    v = str(v).strip()
    
    # 1. วงเล็บเหลี่ยม (Bracketed Formula)
    if v.startswith("[") and v.endswith("]"):
        return "bracketed_formula"
        
    # 2. โครงสร้างซ้อนหรือใส่สารเติมแต่ง (Plus)
    if " + " in v:
        return "multi_material_plus"
        
    # 3. สารผสม (Blends / Mixture) ใช้เครื่องหมาย Colon
    if ":" in v:
        return "multi_material_colon"
        
    # 4. ลบหน่วยวัดทิ้งชั่วคราวเพื่อป้องกัน Slash หลอก
    v_no_units = v.replace("mg/mL", "").replace("w/w", "").replace("v/v", "")
    
    # 5. เช็ค Slash อีกครั้ง (เช่น การเขียนทับศัพท์ Cs/FA/MA หรือ TiO2/SnO2)
    if "/" in v_no_units:
        return "multi_material_slash"
        
    return "plain"

# ทดสอบรันและดูผลลัพธ์
mat = tables["layers"]["material"].dropna().str.strip()
counts = mat.map(classify_material_format).value_counts()
print(counts)

for cat in counts.index:
    print(f"  {cat}: {list(mat[mat.map(classify_material_format) == cat].unique()[:6])}")

material
plain                   609
multi_material_plus      18
multi_material_slash     17
multi_material_colon     10
bracketed_formula         4
Name: count, dtype: int64
  plain: ['ITO', 'NiOx NPs', 'C60', 'BCP', 'Ag', 'EMIMI-1%']
  multi_material_plus: ['PbI2 + PbBr2 + FAI + MABr + CsI', 'Ar plasma + HNO3', 'PTAA + Al2O3', 'BCP + SnO2', 'Ethanolamine + 2PACz', 'RbCl + MeO-4PACz']
  multi_material_slash: ['MgF2/PET/ITO', 'LiF/C60', 'NiOx NPs / Me-4PACz', 'LiF / C60 / BCP', 'FACl / TEACl', 'CsFAMA (Cs/FA/MA triple-cation)']
  multi_material_colon: ['La:NiOx', 'Cu:NiOx', 'Sm:NiOx NPs', 'Sm:NiOx NPs (1.0 mol% Sm)', 'Sm:NiOx NPs (2.0 mol% Sm)', 'Cu:NiO']
  bracketed_formula: ['[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ]']


In [16]:
import re

def extract_site_fractions(part):
    return {name: float(frac) for name, frac in re.findall(r"([A-Za-z]+)(\d*\.\d+|\d+)", part)}

def parse_bracketed_formula(v):
    formula = v.strip().strip("[]").strip()
    m = re.search(r"(Pb|Sn)", formula)
    b_site = m.group(1) if m else None
    a_part = formula[:m.start()] if m else formula
    x_part = formula[m.end():] if m else ""
    return {"mat_1": formula, "b_site": b_site,
            "a_site": extract_site_fractions(a_part),
            "x_site": extract_site_fractions(x_part)}

def parse_plus(v):
    parts = [p.strip() for p in v.split(" + ")]
    return {"mat_1": parts[0], "mat_2": parts[1] if len(parts) > 1 else None}

def parse_colon(v):
    head, _, tail = v.partition("(")
    mats = [m.strip() for m in head.strip().split(":")]
    ratio = None
    rm = re.search(r"(\d+)\s*:\s*(\d+)", tail)
    if rm:
        ratio = f"{rm.group(1)}:{rm.group(2)}"
    return {"mat_1": mats[0], "mat_2": mats[1] if len(mats) > 1 else None, "blend_ratio": ratio}

def parse_slash(v):
    head, _, tail = v.partition("(")
    parts = [p.strip() for p in head.strip().split("/")]
    if len(parts) == 1 and tail:
        parts = [p.strip() for p in tail.rstrip(") ").split("/")]
    return {"mat_1": parts[0], "mat_2": parts[1] if len(parts) > 1 else None,
            "mat_3": parts[2] if len(parts) > 2 else None}

UNIT_SUFFIX = re.compile(r"^(.+?)[\s-]+(\d+(?:\.\d+)?\s*(?:%|mg/mL|mg/ml|mM|wt%))$")
def parse_plain(v):
    m = UNIT_SUFFIX.match(v.strip())
    if m:
        return {"mat_1": m.group(1).strip(), "note": m.group(2).strip()}
    return {"mat_1": v.strip(), "note": None}

def classify_material_format(v):
    v = v.strip()
    if re.search(r"Pb|Sn", v):
        return "bracketed_formula"
    if " + " in v:
        return "multi_material_plus"
    if ":" in v.split("(")[0]:
        return "multi_material_colon"
    dm = re.search(r"\d", v)
    si = v.find("/")
    if si != -1 and (dm is None or si < dm.start()):
        return "multi_material_slash"
    return "plain"

PARSERS = {
    "bracketed_formula": parse_bracketed_formula,
    "multi_material_plus": parse_plus,
    "multi_material_colon": parse_colon,
    "multi_material_slash": parse_slash,
    "plain": parse_plain,
}

def parse_material(v):
    cat = classify_material_format(v)
    result = PARSERS[cat](v)
    result["format"] = cat
    return result

mat = tables["layers"]["material"].dropna().str.strip()
parsed = mat.map(parse_material)
mat_df = pd.json_normalize(parsed)
mat_df.index = parsed.index
mat_df.columns = [c.replace("a_site.", "a_").replace("x_site.", "x_") for c in mat_df.columns]

layers_material_parsed = pd.concat(
    [tables["layers"].loc[mat.index, ["device_id", "layer_type"]], mat_df], axis=1
)

In [17]:
layers_material_parsed.head(3)

,device_id,layer_type,mat_1,note,format,b_site,a_Cs,a_FA,a_MA,x_I,x_Br,x_PbBr,mat_2,blend_ratio,a_Rb,mat_3,x_O
0,10.1002admi.202500231_Sample1,Anode,ITO,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1002admi.202500231_Sample1,HTL,NiOx NPs,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1002admi.202500231_Sample1,PSK,(Cs.05FA.79MA.16)Pb(I.87Br.13)3,NaN,bracketed_formula,Pb,0.05,0.79,0.16,0.87,0.13,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
layers_material_parsed

,device_id,layer_type,mat_1,note,format,b_site,a_Cs,a_FA,a_MA,x_I,x_Br,x_PbBr,mat_2,blend_ratio,a_Rb,mat_3,x_O
0,10.1002admi.202500231_Sample1,Anode,ITO,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1002admi.202500231_Sample1,HTL,NiOx NPs,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1002admi.202500231_Sample1,PSK,(Cs.05FA.79MA.16)Pb(I.87Br.13)3,NaN,bracketed_formula,Pb,0.05,0.79,0.16,0.87,0.13,NaN,NaN,NaN,NaN,NaN,NaN
5,10.1002admi.202500231_Sample1,ETL,C60,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,10.1002admi.202500231_Sample1,ETL treatment,BCP,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,10_31635_ccschem_025_202505977_Sample3,PSK,Cs0.05(FA0.95MA0.05)0.95Pb(I0.95Br0.05)3,NaN,bracketed_formula,Pb,0.05,0.95,0.05,0.95,0.05,NaN,NaN,NaN,NaN,NaN,NaN
756,10_31635_ccschem_025_202505977_Sample3,PSK treatment,PEAI,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
757,10_31635_ccschem_025_202505977_Sample3,ETL,PC61BM,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
758,10_31635_ccschem_025_202505977_Sample3,ETL treatment,BCP,NaN,plain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## F. `stability` (temperature/humidity/retention) — explore

In [19]:
stb0 = tables["stability"]
for col in ["temperature_c", "humidity_pct", "retention_pct"]:
    s = stb0[col].dropna().str.strip()
    bad = s[(s != "") & ~s.map(is_num)]
    print(f"{col}: {len(bad)} แถว parse ตรงๆ ไม่ได้ -> {sorted(bad.unique())}")

temperature_c: 10 แถว parse ตรงๆ ไม่ได้ -> ['20 ± 5', '50 ± 5', 'RT']
humidity_pct: 29 แถว parse ตรงๆ ไม่ได้ -> ['10 ± 5', '25 ± 5', '30 - 50', '30 ± 5', '30-40', '30-50', '30-60', '45–55', '50 - 60', '50 ± 5', '50-60', '50–60', '60 ± 5', '60-70', '< 20']
retention_pct: 2 แถว parse ตรงๆ ไม่ได้ -> ['< 20', '< 55']


In [20]:
stb0 = tables["stability"].copy()
cols = ["temperature_c", "humidity_pct", "retention_pct"]

# ฟังก์ชันเช็คตัวเลขแบบย่อ
def is_num(val):
    try: float(val); return True
    except: return False

# ค้นหาและรวม Index ที่มีปัญหาด้วย Loop
bad_idx = pd.Index([])
for col in cols:
    s = stb0[col].dropna().str.strip()
    bad_idx = bad_idx.union(s[(s != "") & ~s.map(is_num)].index)

print(f"จำนวนแถวที่ Parse ตรงๆ ไม่ได้ทั้งหมด: {len(bad_idx)} แถว")

# สุ่มดูข้อมูล 5 แถว (ถ้ามี)
if not bad_idx.empty:
    display(stb0.loc[bad_idx, ["device_id"] + cols].sample(n=min(5, len(bad_idx))))

จำนวนแถวที่ Parse ตรงๆ ไม่ได้ทั้งหมด: 36 แถว


,device_id,temperature_c,humidity_pct,retention_pct
27,10.1021_acsami.3c02156_Sample4,NaN,60-70,80
75,10_1039_d3lf00258f_Sample1,RT,40,NaN
70,10_1021_acsami_5c15868_Sample3,25,25 ± 5,87.7
6,10.1016_j.apsusc.2021.149973_Sample1,25,30-40,92
60,10_1016_j_solener_2025_113286_Sample2,RT,35,80


## F. `stability` — fix

In [21]:
def parse_temperature(v):
    if not isinstance(v, str) or not v.strip():
        return (None, "")
    s = v.strip()
    if is_num(s):
        return (float(s), "")
    if s.upper() == "RT":
        return (CONFIG["RT_CELSIUS"],
                f"RT (room temperature) -> {CONFIG['RT_CELSIUS']:g} C [assumption: CONFIG]")
    m = re.fullmatch(rf"({NUM})\s*±\s*({NUM})", s)   # เขียนเป็น ค่ากลาง ± คลาดเคลื่อน
    if m:
        c, t = float(m[1]), float(m[2])
        return (c, f"ค่ากลาง±คลาดเคลื่อน: ใช้ {c:g} C (±{t:g})")
    return (None, f"parse ไม่ได้: {s!r}")

def parse_range_pm(v):
    """สำหรับ humidity: ตัวเลข | ช่วง a-b | c ± t -> (parsed, min, max, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    m = re.fullmatch(rf"({NUM})\s*±\s*({NUM})", s)
    if m:
        c, t = float(m[1]), float(m[2])
        return (c, c - t, c + t, f"ค่ากลาง±คลาดเคลื่อน: {c:g} (min={c - t:g}, max={c + t:g})")
    m = re.fullmatch(rf"([<>])\s*({NUM})", s)          # censored เช่น "< 20"
    if m:
        x = float(m[2])
        lo, hi = (0.0, x) if m[1] == "<" else (x, None)
        return (x, lo, hi,
                f"ค่า censored {m[1]}{x:g}: ใช้ {x:g} เป็นขอบ -- ห้ามใช้เป็นค่าจริงโดยไม่ดูขอบเขต")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

def parse_censored(v):
    """สำหรับ retention: ตัวเลข | '< x' -> (parsed, censor, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, "", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "", "")
    m = re.fullmatch(rf"([<>])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1],
                f"ค่า censored: รู้แค่ว่า {'ต่ำ' if m[1] == '<' else 'สูง'}กว่า {m[2]} — ห้ามใช้เป็นค่าจริงโดยไม่ดูธง censor")
    return (None, "", f"parse ไม่ได้: {s!r}")

stb = tables["stability"]

t_ = stb["temperature_c"].map(parse_temperature)
stb["temperature_c_parsed"] = t_.map(lambda x: x[0])
stb["temperature_c_note"] = t_.map(lambda x: x[1])

h = stb["humidity_pct"].map(parse_range_pm)
stb["humidity_pct_parsed"] = h.map(lambda x: x[0])
stb["humidity_pct_min"] = h.map(lambda x: x[1])
stb["humidity_pct_max"] = h.map(lambda x: x[2])
stb["humidity_pct_note"] = h.map(lambda x: x[3])

# t80_hr เดิมว่างทั้งคอลัมน์จึงไม่เคยมี parser -- Raw Data ชุดที่ 2 เริ่มมีค่า (มีทั้ง "> 1300")
t80 = stb["t80_hr"].map(parse_censored)
stb["t80_hr_parsed"] = t80.map(lambda x: x[0])
stb["t80_hr_censor"] = t80.map(lambda x: x[1])
stb["t80_hr_note"] = t80.map(lambda x: x[2])

r = stb["retention_pct"].map(parse_censored)
stb["retention_pct_parsed"] = r.map(lambda x: x[0])
stb["retention_pct_censor"] = r.map(lambda x: x[1])
stb["retention_pct_note"] = r.map(lambda x: x[2])

for col, pcol, note_col in [("temperature_c", "temperature_c_parsed", "temperature_c_note"),
                            ("humidity_pct", "humidity_pct_parsed", "humidity_pct_note"),
                            ("t80_hr", "t80_hr_parsed", "t80_hr_note"),
                            ("retention_pct", "retention_pct_parsed", "retention_pct_note")]:
    hit = stb[note_col] != ""
    for _, row in stb[hit].iterrows():
        audit_rows.append({"table": "stability", "column": col,
                           "device_id": row["device_id"], "raw": row[col],
                           "parsed": row[pcol],
                           "vmin": row.get("humidity_pct_min") if col == "humidity_pct" else None,
                           "vmax": row.get("humidity_pct_max") if col == "humidity_pct" else None,
                           "extra": row.get("retention_pct_censor", "") if col == "retention_pct" else "",
                           "note": row[note_col]})

hit_any = (stb["temperature_c_note"] != "") | (stb["humidity_pct_note"] != "") | (stb["retention_pct_note"] != "")
print(f"แถวที่ถูกตีความ: {int(hit_any.sum())}")
stb.loc[hit_any, ["device_id", "environment", "temperature_c", "temperature_c_parsed",
                  "humidity_pct", "humidity_pct_parsed", "humidity_pct_min", "humidity_pct_max",
                  "retention_pct", "retention_pct_parsed", "retention_pct_censor"]]

from IPython.display import display

print("=== temperature_c ===")
mask_t = stb["temperature_c_note"] != ""
display(stb.loc[mask_t, ["device_id", "temperature_c", "temperature_c_parsed", "temperature_c_note"]])

print("=== humidity_pct ===")
mask_h = stb["humidity_pct_note"] != ""
display(stb.loc[mask_h, ["device_id", "humidity_pct", "humidity_pct_parsed",
                          "humidity_pct_min", "humidity_pct_max", "humidity_pct_note"]])

print("=== retention_pct ===")
mask_r = stb["retention_pct_note"] != ""
display(stb.loc[mask_r, ["device_id", "retention_pct", "retention_pct_parsed",
                          "retention_pct_censor", "retention_pct_note"]])

แถวที่ถูกตีความ: 36
=== temperature_c ===


,device_id,temperature_c,temperature_c_parsed,temperature_c_note
41,10_1002_aenm_202405016_Sample1,50 ± 5,50.0,ค่ากลาง±คลาดเคลื่อน: ใช้ 50 C (±5)
42,10_1002_aenm_202405016_Sample2,50 ± 5,50.0,ค่ากลาง±คลาดเคลื่อน: ใช้ 50 C (±5)
59,10_1016_j_solener_2025_113286_Sample1,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]
60,10_1016_j_solener_2025_113286_Sample2,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]
65,10_1021_acsami_5c13836_Sample1,20 ± 5,20.0,ค่ากลาง±คลาดเคลื่อน: ใช้ 20 C (±5)
66,10_1021_acsami_5c13836_Sample2,20 ± 5,20.0,ค่ากลาง±คลาดเคลื่อน: ใช้ 20 C (±5)
75,10_1039_d3lf00258f_Sample1,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]
76,10_1039_d3lf00258f_Sample2,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]
77,10_1039_d3lf00258f_Sample3,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]
78,10_1039_d3lf00258f_Sample4,RT,25.0,RT (room temperature) -> 25 C [assumption: CONFIG]


=== humidity_pct ===


,device_id,humidity_pct,humidity_pct_parsed,humidity_pct_min,humidity_pct_max,humidity_pct_note
4,10.1016.jpowsour.2020.228738_Sample1,30-60,45.0,30.0,60.0,"ช่วงค่า: ใช้ค่ากลาง 45 (min=30, max=60)"
5,10.1016.jpowsour.2020.228738_Sample2,30-60,45.0,30.0,60.0,"ช่วงค่า: ใช้ค่ากลาง 45 (min=30, max=60)"
6,10.1016_j.apsusc.2021.149973_Sample1,30-40,35.0,30.0,40.0,"ช่วงค่า: ใช้ค่ากลาง 35 (min=30, max=40)"
7,10.1016_j.apsusc.2021.149973_Sample1,50-60,55.0,50.0,60.0,"ช่วงค่า: ใช้ค่ากลาง 55 (min=50, max=60)"
8,10.1016_j.apsusc.2021.149973_Sample2,30-40,35.0,30.0,40.0,"ช่วงค่า: ใช้ค่ากลาง 35 (min=30, max=40)"
9,10.1016_j.apsusc.2021.149973_Sample2,50-60,55.0,50.0,60.0,"ช่วงค่า: ใช้ค่ากลาง 55 (min=50, max=60)"
10,10.1016_j.joule.2022.06.026_Sample1,< 20,20.0,0.0,20.0,ค่า censored <20: ใช้ 20 เป็นขอบ -- ห้ามใช้เป็นค่าจริงโดยไม่ดูขอบเขต
11,10.1016_j.joule.2022.06.026_Sample3,< 20,20.0,0.0,20.0,ค่า censored <20: ใช้ 20 เป็นขอบ -- ห้ามใช้เป็นค่าจริงโดยไม่ดูขอบเขต
13,10.1016_j.joule.2022.06.026_Sample3,30 - 50,40.0,30.0,50.0,"ช่วงค่า: ใช้ค่ากลาง 40 (min=30, max=50)"
19,10.1016j.nanoen.2021.106285_Sample1,50 - 60,55.0,50.0,60.0,"ช่วงค่า: ใช้ค่ากลาง 55 (min=50, max=60)"


=== retention_pct ===


,device_id,retention_pct,retention_pct_parsed,retention_pct_censor,retention_pct_note
30,10.1038_s41598-023-32697-z_Sample1,< 20,20.0,<,ค่า censored: รู้แค่ว่า ต่ำกว่า 20 — ห้ามใช้เป็นค่าจริงโดยไม่ดูธง censor
59,10_1016_j_solener_2025_113286_Sample1,< 55,55.0,<,ค่า censored: รู้แค่ว่า ต่ำกว่า 55 — ห้ามใช้เป็นค่าจริงโดยไม่ดูธง censor


## G. `vendors.purity` — explore

In [22]:
s = tables["vendors"]["purity"].dropna().str.strip()
bad = s[(s != "") & ~s.map(is_num)]
print(f"{len(bad)} แถว ไม่ใช่ตัวเลขล้วน — ปนกันหลายชนิด (qualifier/เกรด/wt%/sheet-resistance/missing):")
bad.value_counts()

69 แถว ไม่ใช่ตัวเลขล้วน — ปนกันหลายชนิด (qualifier/เกรด/wt%/sheet-resistance/missing):


purity
Not provided    17
>97              8
Anhydrous        6
80-90            5
9 - 15 Ω □-1     4
>99.99           4
≥98              4
≥99              4
20 wt            3
> 99             2
> 99.9           2
>99.0            2
≥99.99           2
≥99.5            2
≥99.9            2
99.9%            1
≥ 99.9           1
Name: count, dtype: int64

## G. `vendors.purity` — fix

In [23]:
def parse_purity(v):
    """คืน (pct, qualifier, kind, note)"""
    if not isinstance(v, str) or not v.strip() or v.strip() == "Not provided":
        return (None, "", "missing", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "=", "percent", "")
    m = re.fullmatch(rf"([>≥])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1], "percent_lower_bound",
                f"qualifier {m[1]}: ความบริสุทธิ์อย่างน้อย {m[2]}%")
    m = re.fullmatch(rf"({NUM})\s*%", s)              # เขียนหน่วย % ติดมาด้วย
    if m:
        return (float(m[1]), "=", "percent", f"ตัด '%' ออกจาก {s!r}")
    m = re.fullmatch(rf"([>≥])\s*({NUM})\s*%", s)
    if m:
        return (float(m[2]), m[1], "percent_lower_bound",
                f"qualifier {m[1]} + ตัด '%': ความบริสุทธิ์อย่างน้อย {m[2]}%")
    if s.lower() == "anhydrous":
        return (None, "", "grade_descriptor",
                "เกรดตัวทำละลาย 'ปราศจากน้ำ' — ไม่ใช่ตัวเลขความบริสุทธิ์")
    m = re.fullmatch(rf"({NUM})\s*wt", s)
    if m:
        return (None, "", "concentration_wt_pct",
                f"ความเข้มข้น {m[1]} wt% ของ dispersion — ไม่ใช่ purity (ข้อมูลผิดชนิด)")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})\s*%?", s)   # ช่วงความบริสุทธิ์ เช่น 80-90
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, "~", "percent_range",
                f"ช่วงความบริสุทธิ์ {a:g}-{b:g}% ใช้ค่ากลาง {(a + b) / 2:g}%")
    if "Ω" in s or "Ω" in s or "□" in s:
        return (None, "", "sheet_resistance_misfiled",
                f"sheet resistance ของ substrate ({s}) ถูกกรอกในช่อง purity — Ω ในไฟล์คือ U+2126 OHM SIGN")
    return (None, "", "unrecognized", f"จำแนกไม่ได้: {s!r}")

ven = tables["vendors"]
pu = ven["purity"].map(parse_purity)
ven["purity_pct"] = pu.map(lambda x: x[0])
ven["purity_qualifier"] = pu.map(lambda x: x[1])
ven["purity_kind"] = pu.map(lambda x: x[2])
_purity_note = pu.map(lambda x: x[3])

hit = ven["purity"].notna() & (ven["purity"].str.strip() != "") & ~ven["purity"].fillna("").map(is_num)
for idx in ven.index[hit]:
    audit_rows.append({"table": "vendors", "column": "purity",
                       "device_id": ven.at[idx, "device_id"], "raw": ven.at[idx, "purity"],
                       "parsed": ven.at[idx, "purity_pct"], "vmin": None, "vmax": None,
                       "extra": ven.at[idx, "purity_kind"], "note": _purity_note[idx]})

print("จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):")
display(ven.loc[ven["purity"].notna() & (ven["purity"].str.strip() != ""), "purity_kind"]
           .value_counts().rename("n_rows").to_frame())

print("แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):")
ven.loc[hit, ["device_id", "material", "purity", "purity_pct",
              "purity_qualifier", "purity_kind"]].drop_duplicates(
    subset=["material", "purity", "purity_kind"]).reset_index(drop=True).rename(
    columns={"purity": "before (raw)", "purity_pct": "after (pct)"})

# ---- materials.amount: ค่าที่ไม่ใช่สเกลาร์ (มากับ Raw Data ชุดที่ 2) ----
# เจอ "1:50" หน่วย "v:v" = อัตราส่วนการเจือจาง ไม่ใช่ปริมาณ -- แปลงเป็นตัวเลขเดี่ยวไม่ได้
#   เก็บเป็นอัตราส่วนแยกคอลัมน์ แล้วปล่อย amount_parsed เป็น NaN พร้อม note
#   (ห้ามปล่อยเงียบ ๆ เพราะด่านตรวจช่วง 2 จะนับเป็น "หลุด")
def parse_amount(v, u):
    """คืน (parsed, ratio, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, "")
    s = v.strip()
    if is_num(s):
        return (float(s), None, "")
    m = re.fullmatch(rf"({NUM})\s*[:/]\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return (None, round(a / b, 6),
                f"อัตราส่วน {s} (หน่วย {u!r}) ไม่ใช่ปริมาณสเกลาร์ -- เก็บเป็น amount_ratio = {a / b:g}")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, None, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    return (None, None, f"parse ไม่ได้: {s!r}")

_mt = tables["materials"]
_am = [parse_amount(v, u) for v, u in zip(_mt["amount"], _mt["unit"])]
_mt["amount_parsed"] = [t[0] for t in _am]
_mt["amount_ratio"] = [t[1] for t in _am]
_mt["amount_note"] = [t[2] for t in _am]

_hit = _mt["amount_note"] != ""
for _, r in _mt[_hit].iterrows():
    audit_rows.append({"table": "materials", "column": "amount",
                       "device_id": r["device_id"], "raw": r["amount"],
                       "parsed": r["amount_parsed"], "vmin": None, "vmax": None,
                       "extra": r["unit"], "note": r["amount_note"]})
print(f"materials.amount ที่ต้องตีความ: {int(_hit.sum())} แถว")
if _hit.any():
    display(_mt.loc[_hit, ["device_id", "material", "amount", "unit",
                           "amount_parsed", "amount_ratio", "amount_note"]])

จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):


,n_rows
purity_kind,
percent,596
percent_lower_bound,33
missing,17
grade_descriptor,6
percent_range,5
sheet_resistance_misfiled,4
concentration_wt_pct,3


แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):
materials.amount ที่ต้องตีความ: 2 แถว


,device_id,material,amount,unit,amount_parsed,amount_ratio,amount_note
443,10.1021_acsami.3c02156_Sample2,Ethanolamine,1:50,v:v,NaN,0.02,อัตราส่วน 1:50 (หน่วย 'v:v') ไม่ใช่ปริมาณสเกลาร์ -- เก็บเป็น amount_ratio = 0.02
467,10.1021_acsami.3c02156_Sample4,Ethanolamine,1:50,v:v,NaN,0.02,อัตราส่วน 1:50 (หน่วย 'v:v') ไม่ใช่ปริมาณสเกลาร์ -- เก็บเป็น amount_ratio = 0.02


## H. จับกลุ่ม material family — explore

ชื่อวัสดุปรากฏใน 3 ตาราง (`layers`, `materials`, `vendors`) เช่น `NiOx / NiOx NPs / NiOx Nanoparticles` = ตระกูลเดียวกัน (NiOx)

In [24]:
def material_family(x):
    s = str(x).lower().strip()
    if "nio" in s: return "NiOx"
    if re.search(r"\bc60\b", s): return "C60"                     # รวมชั้นซ้อน เช่น LiF/C60
    if "bcp" in s: return "BCP"
    if "pbi2" in s: return "PbI2"
    if "pedot" in s: return "PEDOT:PSS"
    if "spiro" in s: return "Spiro-OMeTAD"
    if any(k in s for k in ["pacz","2pacz","me-2pacz","mecz"]): return "SAM (xPACz)"
    if any(k in s for k in ["pcbm", "pc61bm", "pc60bm"]): return "PCBM"
    if "sno2" in s or "sno" == s: return "SnO2"                  # ETL อนินทรีย์ (ชุดที่ 2)
    if re.search(r"\b[if]to\b", s): return "TCO"                  # รวม MgF2/PET/ITO, ITO-PET
    if s in ("ag", "au", "cu", "cr", "cr + au", "cu target"): return "Metal electrode"
    if any(k in s for k in ["peai", "pdi", "pei"]): return "Amine passivation"   # PEAI/PDI/CF3-PEAI/PEI = สาร passivation ตระกูลเอมีน
    return None

mats_preview = pd.concat([tables["layers"][["material"]].assign(src="layers"),
                          tables["materials"][["material"]].assign(src="materials"),
                          tables["vendors"][["material"]].assign(src="vendors")], ignore_index=True)
mats_preview = mats_preview.dropna()
mats_preview["material"] = mats_preview["material"].astype(str).str.strip()
mats_preview["family"] = mats_preview["material"].map(material_family)
fam_preview = (mats_preview.dropna(subset=["family"]).groupby("family")["material"]
                  .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v,c in Counter(s).most_common()))
                  .reset_index(name="spellings_found"))
fam_preview["n_spellings"] = fam_preview["spellings_found"].str.count(r"\|") + 1
print("ตระกูลวัสดุที่มีหลายชื่อ (ยังไม่ apply เป็นคอลัมน์):")
fam_preview.sort_values("n_spellings", ascending=False)

ตระกูลวัสดุที่มีหลายชื่อ (ยังไม่ apply เป็นคอลัมน์):


,family,spellings_found,n_spellings
4,NiOx,NiOx ×90 | NiOx NPs ×85 | NiO target ×10 | NiOx Nanoparticles ×5 | NiOx nanoparticles ×5 | NiOx (sputtered...,19
7,SAM (xPACz),Me-4PACz ×16 | 2PACz ×13 | MeO-4PACz ×6 | Me-2PACz ×4 | RbCl + MeO-4PACz ×2 | Me-2PACz (SAM) ×2 | Ethano...,14
9,TCO,ITO ×96 | FTO ×77 | ITO coated glass ×33 | FTO glass ×19 | FTO coated glass ×15 | ITO-PET ×10 | ITO-PET ...,14
0,Amine passivation,PEAI ×60 | PEI ×12 | PDI ×6 | CF3-PEAI ×4 | PDI (piperazine dihydriodide) ×3 | CF3-PEAI/MAI ×2,6
5,PCBM,PC61BM ×73 | PCBM ×59 | PC60BM ×12 | PCBM + TBABF4 ×4 | PC61BM (PCBM) ×4 | PCBM ([60]PCBM) ×2,6
2,C60,"C60 ×106 | Fullerene-C60 ×5 | CsI, FAI, MAI, MACl, PEAI, C60 ×5 | LiF/C60 ×4 | LiF / C60 / BCP ×3 | PCBM:C...",6
3,Metal electrode,Ag ×238 | Cu ×30 | Cr ×4 | Au ×4 | Cr + Au ×2 | Cu target ×1,6
1,BCP,"BCP ×225 | BCP (2,9-Dimethyl-4,7-diphenyl-1,10-phenanthroline) ×5 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | ...",5
6,PbI2,PbI2 ×170 | Cs0.1FA0.9PbI2.855Br0.145 ×6 | PbI2 + PbBr2 + FAI + MABr + CsI ×1,3
8,SnO2,SnO2 ×11 | SnO2-A dispersion ×5,2


## H. material family -- fix

จับด้วย keyword ในชื่อ ไม่ใช่ exact-match แบบ C เพราะ `NiOx`/`NiOx NPs`/`NiOx Nanoparticles` เป็นคนละ string กันจริง ๆ

**7 ตระกูลที่ครอบคลุม**: NiOx, C60, BCP, PbI2, PEDOT:PSS, Spiro-OMeTAD, SAM (xPACz)

เพิ่มคอลัมน์ใหม่ `material_family` ให้ `layers`/`materials`/`vendors` -- ไม่ทับ `material` เดิม

NaN เยอะเพราะสารตั้งต้น/ตัวทำละลาย (PbBr2, DMF, CsI, NaOH, ...) ไม่อยู่ใน 7 ตระกูลนี้ -- ครอบคลุมเฉพาะชั้นฟังก์ชัน (HTL/ETL) ที่มักถูกเขียนชื่อหลายแบบ

In [25]:
for name in ["layers", "materials", "vendors"]:
    tables[name]["material_family"] = tables[name]["material"].map(material_family)

mats = pd.concat([tables[n][["material", "material_family"]].assign(src=n)
                  for n in ["layers", "materials", "vendors"]], ignore_index=True)
classified = mats.dropna(subset=["material_family"])
fam = (classified.groupby("material_family")["material"]
                  .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v, c in Counter(s).most_common()))
                  .reset_index(name="spellings_found"))
fam["n_rows"] = classified.groupby("material_family").size().values
print(f"จัดกลุ่มได้ {len(classified)} / {len(mats)} แถว (รวม 3 ตาราง) เข้า {fam['material_family'].nunique()} family")

display(fam.sort_values("n_rows", ascending=False))

จัดกลุ่มได้ 1617 / 3516 แถว (รวม 3 ตาราง) เข้า 10 family


,material_family,spellings_found,n_rows
3,Metal electrode,Ag ×238 | Cu ×30 | Cr ×4 | Au ×4 | Cr + Au ×2 | Cu target ×1,279
9,TCO,ITO ×96 | FTO ×77 | ITO coated glass ×33 | FTO glass ×19 | FTO coated glass ×15 | ITO-PET ×10 | ITO-PET ...,274
1,BCP,"BCP ×225 | BCP (2,9-Dimethyl-4,7-diphenyl-1,10-phenanthroline) ×5 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | ...",238
4,NiOx,NiOx ×90 | NiOx NPs ×85 | NiO target ×10 | NiOx Nanoparticles ×5 | NiOx nanoparticles ×5 | NiOx (sputtered...,216
6,PbI2,PbI2 ×170 | Cs0.1FA0.9PbI2.855Br0.145 ×6 | PbI2 + PbBr2 + FAI + MABr + CsI ×1,177
5,PCBM,PC61BM ×73 | PCBM ×59 | PC60BM ×12 | PCBM + TBABF4 ×4 | PC61BM (PCBM) ×4 | PCBM ([60]PCBM) ×2,154
2,C60,"C60 ×106 | Fullerene-C60 ×5 | CsI, FAI, MAI, MACl, PEAI, C60 ×5 | LiF/C60 ×4 | LiF / C60 / BCP ×3 | PCBM:C...",125
0,Amine passivation,PEAI ×60 | PEI ×12 | PDI ×6 | CF3-PEAI ×4 | PDI (piperazine dihydriodide) ×3 | CF3-PEAI/MAI ×2,87
7,SAM (xPACz),Me-4PACz ×16 | 2PACz ×13 | MeO-4PACz ×6 | Me-2PACz ×4 | RbCl + MeO-4PACz ×2 | Me-2PACz (SAM) ×2 | Ethano...,51
8,SnO2,SnO2 ×11 | SnO2-A dispersion ×5,16


In [26]:
print("before (material) -> after (material_family) รายค่า:")
classified.drop_duplicates(subset=["material", "material_family"]).sort_values(
    ["material_family", "material"]).rename(
    columns={"material": "before", "material_family": "after"})[["before", "after", "src"]]

before (material) -> after (material_family) รายค่า:


,before,after,src
1909,CF3-PEAI,Amine passivation,materials
724,CF3-PEAI/MAI,Amine passivation,layers
644,PDI,Amine passivation,layers
3277,PDI (piperazine dihydriodide),Amine passivation,vendors
180,PEAI,Amine passivation,layers
694,PEI,Amine passivation,layers
6,BCP,BCP,layers
2656,"BCP (2,9-Dimethyl-4,7-diphenyl-1,10-phenanthroline)",BCP,vendors
2963,BCP (bathocuproine),BCP,vendors
214,BCP + SnO2,BCP,layers


### ควรยึด src ไหนเป็นตัวตั้งของ `material_family`

**ยึด `layers` เป็นตัวตั้ง** -- อีก 2 ตารางปล่อยให้ตกเป็นผลพลอยได้

| src | คืออะไร | ใช้ตัดสิน family ได้ไหม |
|---|---|---|
| `layers` | สแต็กจริงของอุปกรณ์ (มี `layer_type`: HTL/ETL/PSK/Anode/Cathode) | **ใช่** -- มีบริบทหน้าที่ชัด + join กับ `device_id` ไปหา PCE/grain size ได้ |
| `materials` | ส่วนผสมของแต่ละชั้น (มี `type`: Precursor/Solution/Additive/Antisolvent) | ไม่ -- มิติที่ถูกคือ **role** ซึ่งมีอยู่แล้วใน `type` |
| `vendors` | รายการซื้อของ (ยี่ห้อ/ความบริสุทธิ์) | ไม่ -- เป็น metadata การจัดซื้อ ไม่บอกโครงสร้างอุปกรณ์ |

**เหตุผล**: ชื่อไม่ซ้ำมี 65 (layers) / 61 (materials) / 79 (vendors) แต่ **65 ชื่อที่โผล่เฉพาะใน materials+vendors ไม่เคยเป็นชั้นในสแต็กเลย** (`DMF`, `DMSO`, `CB`, `Chlorobenzene`, `CsBr`, `Acetonitrile`, `Cu target`, ...) -- เป็นตัวทำละลาย/สารตั้งต้น การไล่หา family ให้พวกนี้ไม่มีความหมาย

**วิธีปฏิบัติ**
1. ยังคง apply `material_family` ทั้ง 3 ตารางเหมือนเดิม (ไม่เสียหาย + ช่วยตอนเช็ค cross-table consistency)
2. แต่ **เพิ่มกฎ family ใหม่โดยดู gap จาก `layers` เท่านั้น** = มุมมอง [2] ในเซลล์ถัดไป
3. **อย่าใช้ % coverage รวม 3 ตารางเป็นตัวชี้วัด** -- `vendors` จะไม่มีวันถึง 100% และไม่ควรถึง

**ตัวชี้วัดที่ควรใช้**: coverage ของ `layers` อย่างเดียว -- ตอนนี้ 226/315 = 71.7%

In [27]:
parts = []
for n in ["layers", "materials", "vendors"]:
    d = tables[n].dropna(subset=["material"]).copy()
    d["material"] = d["material"].astype(str).str.strip()
    d["src"] = n
    # บริบทของแต่ละตาราง: layers ใช้ layer_type, materials ใช้ type (role ที่มีในข้อมูลอยู่แล้ว)
    d["context"] = d["layer_type"] if n == "layers" else (d["type"] if n == "materials" else "ซื้อ/vendor")
    parts.append(d[["src", "context", "material"]])

mats = pd.concat(parts, ignore_index=True)
mats["material_family"] = mats["material"].map(material_family)
mats["context"] = mats["context"].fillna("(ไม่ระบุ)")
gap = mats[mats["material_family"].isna()]

print("=== [1] ความครอบคลุมรายตาราง ===")
cov = mats.groupby("src")["material_family"].agg(n_rows="size", n_family=lambda s: s.notna().sum())
cov["n_NaN"] = cov["n_rows"] - cov["n_family"]
cov["pct_covered"] = (cov["n_family"] / cov["n_rows"] * 100).round(1)
display(cov)

print("=== [2] layers ที่ยังไม่มี family -> ควรเพิ่ม (เป็นชั้นฟังก์ชัน) ===")
display(gap[gap["src"] == "layers"].groupby("context")["material"]
        .agg(n="size", names=lambda s: ", ".join(sorted(set(s))[:5]))
        .sort_values("n", ascending=False))

print("=== [3] materials/vendors ที่ยังไม่มี family -> เป็น role ไม่ใช่ family ===")
display(gap[gap["src"] != "layers"].groupby("context")["material"]
        .agg(n="size", names=lambda s: ", ".join(sorted(set(s))[:5]))
        .sort_values("n", ascending=False))

=== [1] ความครอบคลุมรายตาราง ===


,n_rows,n_family,n_NaN,pct_covered
src,,,,
layers,658,509,149,77.4
materials,1113,587,526,52.7
vendors,1551,521,1030,33.6


=== [2] layers ที่ยังไม่มี family -> ควรเพิ่ม (เป็นชั้นฟังก์ชัน) ===


,n,names
context,,
PSK,89,"(FA0.83MA0.17)0.95Cs0.05Pb(I0.9Br0.1)3, (FA0.83MA0.17)0.95Pb(I0.9Br0.1)3, (FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3, ..."
HTL treatment,42,"2Br-4PADMAc, 3F-PTES, Al2O3, Ar plasma, Ar plasma + HNO3"
PSK treatment,11,"BzMIMBr, FACl, FACl / TEACl, MABr, MACl"
ETL treatment,5,"SnOx, ZnO NPs"
ETL2,2,ZnO NPs


=== [3] materials/vendors ที่ยังไม่มี family -> เป็น role ไม่ใช่ family ===


,n,names
context,,
ซื้อ/vendor,1030,"1,2-Dichlorobenzene (DCB), 1-bromobutane, 1-methylimidazole, 2Br-4PADMAc, 2ME (2-methoxyethanol)"
Precursor,330,"Cs0.05FA0.81MA0.14PbBr0.14I2.86, Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3, Cs0.1(FA0.83MA0.17)0.9Pb(I0.85Br0.15)3, C..."
Additive,68,"BMIMBF4, CsI, Ethanolamine, Ethylenediamine, KI"
Solution,57,"Al2O3 (20 wt%), BDT-POZ, BMIMBF4, BMIMBr, BMIMGly"
(ไม่ระบุ),28,"3F-PTES, Ar plasma, BA, BzMIMBr, CsI"
Nanoparticles,9,"BN, CsPbI3 PeQDs, ZnO NPs"
Antisolvent,8,"CB, Chlorobenzene, Ethyl acetate"
Dopant,6,"Cu(CH3COO)2·H2O, La(NO3)2·6H2O, Sm(NO3)3·6H2O"
Surface modifier,4,"LiPF6, NH4PF6, NaPF6, TFFA"


In [28]:
# แสดงชื่อตระกูล พร้อมจำนวนแถวที่จัดกลุ่มได้ เรียงจากมากไปน้อย
print("สรุปจำนวนข้อมูลในแต่ละตระกูลวัสดุ:")
family_counts = classified["material_family"].value_counts().reset_index()
family_counts.columns = ["material_family", "count"] # เปลี่ยนชื่อคอลัมน์ให้เข้าใจง่าย
display(family_counts)

สรุปจำนวนข้อมูลในแต่ละตระกูลวัสดุ:


,material_family,count
0,Metal electrode,279
1,TCO,274
2,BCP,238
3,NiOx,216
4,PbI2,177
5,PCBM,154
6,C60,125
7,Amine passivation,87
8,SAM (xPACz),51
9,SnO2,16


In [29]:
# ที่มาของ `mats`: เซลล์ก่อนหน้า concat 3 ตาราง (layers + materials + vendors) เข้าด้วยกัน
# => ตัวเลขในเซลล์นี้เป็น "รวม 3 ตาราง" คนละฐานกับ coverage 250/315 ที่นับเฉพาะ layers
#
# NaN ตรงนี้ = material_family() คืน None (ไม่ตรงกฎ) ไม่ใช่ข้อมูลขาดหาย -- คอลัมน์ material มีค่าครบ
# และ NaN ส่วนใหญ่ "ถูกต้องแล้ว" เพราะ materials/vendors เต็มไปด้วย solvent/precursor
# (DMF, DMSO, CsI, FAI) ซึ่งไม่ใช่ชั้นฟังก์ชัน จึงไม่ควรมี family ตั้งแต่ต้น
# ตัวที่ต้องสนใจจริงคือ NaN ฝั่ง layers เท่านั้น -> ดูมุมมอง [2] ในเซลล์ก่อนหน้า

nan_mats = mats.loc[mats["material_family"].isna(), "material"]
n_fam = mats["material_family"].nunique()
by_src = mats.assign(no_fam=mats["material_family"].isna()).groupby("src")["no_fam"].agg(["sum", "size"])

print(f"[ฐาน: รวม 3 ตาราง layers+materials+vendors] "
      f"{len(nan_mats)}/{len(mats)} แถว ไม่เข้า {n_fam} ตระกูล ({nan_mats.nunique()} ชื่อไม่ซ้ำ)")
for src, r in by_src.iterrows():
    note = "<-- ตัวนี้เท่านั้นที่ควรไล่เพิ่มกฎ" if src == "layers" else "(solvent/precursor -- ไม่ควรมี family)"
    print(f"    {src:10s} ไม่มี family {int(r['sum']):4d}/{int(r['size']):4d} แถว  {note}")

gap = mats[mats["material_family"].isna()]
for src in ["layers", "materials", "vendors"]:
    g = gap[gap["src"] == src]
    print(f"\n--- ตัวอย่างแถวที่ไม่มี family: {src} ({len(g)} แถว) ---")
    display(g.groupby("context")["material"]
             .agg(n="size", ตัวอย่าง=lambda s: ", ".join(sorted(set(s))[:4]))
             .sort_values("n", ascending=False).head(6))

[ฐาน: รวม 3 ตาราง layers+materials+vendors] 1705/3322 แถว ไม่เข้า 10 ตระกูล (170 ชื่อไม่ซ้ำ)
    layers     ไม่มี family  149/ 658 แถว  <-- ตัวนี้เท่านั้นที่ควรไล่เพิ่มกฎ
    materials  ไม่มี family  526/1113 แถว  (solvent/precursor -- ไม่ควรมี family)
    vendors    ไม่มี family 1030/1551 แถว  (solvent/precursor -- ไม่ควรมี family)

--- ตัวอย่างแถวที่ไม่มี family: layers (149 แถว) ---


,n,ตัวอย่าง
context,,
PSK,89,"(FA0.83MA0.17)0.95Cs0.05Pb(I0.9Br0.1)3, (FA0.83MA0.17)0.95Pb(I0.9Br0.1)3, (FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3, ..."
HTL treatment,42,"2Br-4PADMAc, 3F-PTES, Al2O3, Ar plasma"
PSK treatment,11,"BzMIMBr, FACl, FACl / TEACl, MABr"
ETL treatment,5,"SnOx, ZnO NPs"
ETL2,2,ZnO NPs



--- ตัวอย่างแถวที่ไม่มี family: materials (526 แถว) ---


,n,ตัวอย่าง
context,,
Precursor,330,"Cs0.05FA0.81MA0.14PbBr0.14I2.86, Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3, Cs0.1(FA0.83MA0.17)0.9Pb(I0.85Br0.15)3, CsBr"
Additive,68,"BMIMBF4, CsI, Ethanolamine, Ethylenediamine"
Solution,57,"Al2O3 (20 wt%), BDT-POZ, BMIMBF4, BMIMBr"
(ไม่ระบุ),28,"3F-PTES, Ar plasma, BA, BzMIMBr"
Nanoparticles,9,"BN, CsPbI3 PeQDs, ZnO NPs"
Antisolvent,8,"CB, Chlorobenzene, Ethyl acetate"



--- ตัวอย่างแถวที่ไม่มี family: vendors (1030 แถว) ---


,n,ตัวอย่าง
context,,
ซื้อ/vendor,1030,"1,2-Dichlorobenzene (DCB), 1-bromobutane, 1-methylimidazole, 2Br-4PADMAc"


In [30]:
print("\n--- ชื่อที่พบบ่อยที่สุด (รวมทุกตาราง) ---")
print(nan_mats.value_counts().head(20))


--- ชื่อที่พบบ่อยที่สุด (รวมทุกตาราง) ---
material
CsI                                         174
FAI                                         170
MABr                                        130
PbBr2                                       122
DMF                                          86
DMSO                                         83
IPA                                          59
MACl                                         55
MAI                                          35
Chlorobenzene                                34
CB                                           27
NaOH                                         23
LiF                                          20
Ethylenediamine                              15
Ethanolamine                                 14
NMP                                          14
Toluene                                      14
Ni(NO3)2·6H2O                                13
Methanol                                     13
Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.

### `material_family` เป็น NaN แล้วใช้ต่อได้ไหม

**ได้ -- NaN ไม่ใช่อุปสรรค** เพราะ `material_family` ไม่ใช่ตัวแปรที่ใช้ตอบคำถามวิจัย มันเป็นแค่ **ตัวควบคุม (covariate)** ที่ต้องสะอาดไว้ ไม่ใช่ตัวขับผลลัพธ์

**เหตุผล 3 ข้อ**

1. ชื่อที่เหลือ (39 ชื่อ) เกือบทั้งหมดอยู่ใน `HTL treatment` และแต่ละชื่ออยู่ใน **paper เดียวเท่านั้น** -- ตั้ง family ให้ก็ใช้ไม่ได้ เพราะพอทำ Leave-One-Publication-Out (group = DOI) ตอน hold out paper นั้นออก โมเดลจะไม่เคยเห็น category นั้นมาก่อน

2. เวลาสร้าง feature ระดับอุปกรณ์ **ไม่ต้องรู้ว่าใช้สารตัวไหน** รู้แค่ "มี treatment หรือไม่" ก็พอ -- NaN จึงแปลงเป็น `0` ได้ตรง ๆ ไม่ใช่ค่าที่หายไป

   | feature | มี | ไม่มี | ใช้ได้ |
   |---|---|---|---|
   | `has_htl_treatment` | 29 | 15 | ใช้ได้ |
   | `has_psk_passivation` | 20 | 24 | ใช้ได้ |
   | `has_etl_treatment` | 42 | 2 | ทิ้ง -- แทบไม่มี variance |

3. องค์ประกอบของชั้น PSK (44 แถวที่ family เป็น NaN) มีตัวแทนที่ดีกว่าอยู่แล้ว คือ `a_Cs`/`a_FA`/`a_MA`/`x_I`/`x_Br` จากหัวข้อ J ซึ่งเป็น**ตัวเลขต่อเนื่อง** ให้ข้อมูลมากกว่า label เดี่ยว

**แล้วอะไรคือตัวที่ใช้หา process จริง**

ไม่ใช่ตารางนี้ -- อยู่ที่ `process` (`SpinCoating` 204, `Annealing` 161, `AntiSolvent` 35 แถว) ซึ่งค่าตัวเลขจริง (รอบต่อนาที, อุณหภูมิ, เวลา) ยังฝังเป็น free text ใน `description`/`parameter` **ยังไม่ได้ parse** -- นั่นคืองานที่เหลือ

สรุป: `material_family` ทำถึง 79.4% (เฉพาะ `layers`) แล้วหยุด ถูกต้องแล้ว ที่เหลือแปลงเป็น binary flag ตอนสร้างตาราง feature

## J. สัดส่วน A/B/X ของชั้น PSK -- รวม 2 แหล่ง

ชั้น PSK มีข้อมูลองค์ประกอบอยู่ 2 ที่ ใช้เสริมกันได้พอดี:

| แหล่ง | ที่มา | ครอบคลุม |
|---|---|---|
| **สูตรนาม** (หลัก) | `layers.material` เช่น `Cs0.05FA0.85MA0.1PbI3` | 42/44 -- เป็นค่าที่ paper ประกาศเอง |
| **สารตั้งต้น** (เติมช่องว่าง) | `materials` join ด้วย `device_id`+`layer_id` แล้วคำนวณกลับ | 19/44 -- ครอบคลุม 2 แถวที่สูตรแตกไม่ได้พอดี |

รวมกันได้ **44/44 devices** คอลัมน์ `comp_source` บอกว่าค่าของแถวนั้นมาจากแหล่งไหน

**กติกาฝั่งสารตั้งต้น** (ตั้งเข้มไว้ -- ยอมได้น้อยดีกว่าได้เลขผิด)
1. แปลงเป็นโมล: หน่วยตระกูล `molar` (M, mol, mmol) ใช้ตรง ๆ / ตระกูล `mass` (mg, g) หารด้วย molar mass
2. **บังคับให้ทั้งอุปกรณ์ใช้หน่วยตระกูลเดียวกัน** -- ปน `mg` กับ `mol%` แล้วสเกลเพี้ยนหนัก
3. **ตัดเฉพาะสารระเหย (`MACl`) ตามเคมี ไม่ใช่ตามป้าย `type`** -- MACl ระเหยตอน anneal จึงไม่อยู่ในสูตรนาม แต่ `CsI` ไม่ระเหย ต้องนับเสมอ แม้บาง paper จะติดป้ายว่า `Additive`
4. ถ้ามีสารตั้งต้นแถวไหนแปลงไม่ได้ **ไม่คำนวณเลย** แล้วรายงานว่าตกหล่นตัวไหน (`precursor_status`) -- กันเคสที่สัดส่วนถูกปั่นให้ผิดแบบเงียบ ๆ (เจอจริง: `MABr` ไม่มีหน่วย ทำให้ FA พุ่งจาก 0.79 -> 0.94)

**`cross_check`**: 17 devices ที่มีทั้ง 2 แหล่ง เอามาเทียบกันเอง (เกณฑ์ +-0.03) -- ตรง 13 / ต่าง 2 / สูตรแตกไม่ได้ 2 เคสที่ต่างเป็นข้อมูลขัดกันในต้นฉบับจริง (เช่น `solr_202400534` สูตรบอก Br 0.17 แต่รายการสารตั้งต้นไม่มี PbBr2 เลย) เก็บไว้ตรวจมือ ไม่ดันให้ตรง

In [31]:
# ---- J1. แหล่งที่ 1: อ่านสัดส่วนจากสูตรที่เขียนในชื่อชั้น (layers.material) ----
CATIONS = ["FA", "MA", "Cs", "Rb", "Pb", "Sn", "Br", "Cl", "I", "K"]   # เรียงยาว->สั้น: กัน FA ถูกอ่านเป็น F+A
TOKEN = re.compile("(" + "|".join(CATIONS) + r")(\d*\.?\d+)")

def _site_from_str(part):
    """อ่านสัดส่วนไอออนจากสตริง รองรับวงเล็บที่มีตัวคูณต่อท้าย เช่น (FA0.83MA0.17)0.95"""
    out = {}
    for inner, scale in re.findall(r"\(([^)]*)\)\s*(\d*\.?\d+)?", part):
        k = float(scale) if scale else 1.0
        for ion, frac in TOKEN.findall(inner):
            out[ion] = out.get(ion, 0) + float(frac) * k
    rest = re.sub(r"\([^)]*\)\s*\d*\.?\d*", "", part)      # ส่วนที่อยู่นอกวงเล็บ
    for ion, frac in TOKEN.findall(rest):
        out[ion] = out.get(ion, 0) + float(frac)
    return out

def composition_from_formula(f):
    """คืน (A_dict, X_dict, b_site) จากสูตรนาม -- None ถ้าไม่มีตัวเลขสัดส่วนให้แตก"""
    s = str(f).strip().strip("[]").strip()
    m = re.search(r"(Pb|Sn)", s)
    if not m:
        return None
    A, X = _site_from_str(s[:m.start()]), _site_from_str(s[m.end():])
    sA, sX = sum(A.values()), sum(X.values())
    if not sA or not sX:
        return None
    return ({k: v / sA for k, v in A.items()}, {k: v / sX for k, v in X.items()}, m.group(1))

# ทดสอบกับสูตรจริง 2 แบบ: มีวงเล็บซ้อนตัวคูณ / เขียนเรียงตรง ๆ
for _t in ["[(FA0.83MA0.17)0.95Cs0.05Pb(I0.9Br0.1)3]", "Cs0.15FA0.81MA0.04Pb(I2.86Br0.14)3"]:
    _A, _X, _b = composition_from_formula(_t)
    print(_t)
    print("   A:", {k: round(v, 3) for k, v in _A.items()},
          "  B:", _b,
          "  X:", {k: round(v, 3) for k, v in _X.items()})

[(FA0.83MA0.17)0.95Cs0.05Pb(I0.9Br0.1)3]
   A: {'FA': 0.788, 'MA': 0.162, 'Cs': 0.05}   B: Pb   X: {'I': 0.9, 'Br': 0.1}
Cs0.15FA0.81MA0.04Pb(I2.86Br0.14)3
   A: {'Cs': 0.15, 'FA': 0.81, 'MA': 0.04}   B: Pb   X: {'I': 0.953, 'Br': 0.047}


In [32]:
# ---- J2. แหล่งที่ 2 (เตรียม): ตาราง molar mass + ไอออนที่แต่ละสารตั้งต้นให้ ----
MW = {"CsI": 259.81, "CsBr": 212.81, "FAI": 171.97, "MAI": 158.97, "MABr": 111.97,
      "MACl": 67.52, "PbI2": 461.01, "PbBr2": 367.01, "PbCl2": 278.11}
ION = {"CsI":  {"A": {"Cs": 1}, "X": {"I": 1}},   "CsBr": {"A": {"Cs": 1}, "X": {"Br": 1}},
       "FAI":  {"A": {"FA": 1}, "X": {"I": 1}},   "MAI":  {"A": {"MA": 1}, "X": {"I": 1}},
       "MABr": {"A": {"MA": 1}, "X": {"Br": 1}},  "MACl": {"A": {"MA": 1}, "X": {"Cl": 1}},
       "PbI2": {"B": {"Pb": 1}, "X": {"I": 2}},   "PbBr2":{"B": {"Pb": 1}, "X": {"Br": 2}},
       "PbCl2":{"B": {"Pb": 1}, "X": {"Cl": 2}}}
MOLAR = {"m": 1.0, "mol": 1.0, "m total": 1.0, "mmol": 0.001}   # ~ สัดส่วนโมลโดยตรง
MASS  = {"mg": 0.001, "g": 1.0, "g / 2ml": 1.0}                 # ต้องหารด้วย MW
VOLATILE = {"MACl"}   # ระเหยตอน anneal -> ไม่อยู่ในฟิล์ม (CsI ไม่ระเหย ต้องนับ แม้ป้ายจะเขียน Additive)

def canon_precursor(s):
    """ชื่อสารตั้งต้นในไฟล์เขียนไม่เหมือนกัน -> map เข้าชื่อมาตรฐานใน MW"""
    k = re.sub(r"[^a-z0-9]", "", str(s).lower())
    for n in MW:
        if re.sub(r"[^a-z0-9]", "", n.lower()) == k:
            return n
    return None

def unit_family(u):
    """หน่วยแบ่งเป็น 2 ตระกูล -- ปนกันแล้วสเกลเพี้ยน จึงต้องรู้ว่าแถวไหนอยู่ตระกูลไหน"""
    u = str(u).strip().lower()
    return "molar" if u in MOLAR else ("mass" if u in MASS else None)

print("สารตั้งต้นที่รู้จัก:", ", ".join(MW))
print("หน่วยตระกูล molar:", ", ".join(MOLAR))
print("หน่วยตระกูล mass :", ", ".join(MASS))

สารตั้งต้นที่รู้จัก: CsI, CsBr, FAI, MAI, MABr, MACl, PbI2, PbBr2, PbCl2
หน่วยตระกูล molar: m, mol, m total, mmol
หน่วยตระกูล mass : mg, g, g / 2ml


In [33]:
# ---- J3. join ชั้น PSK เข้ากับสารตั้งต้นของชั้นเดียวกัน (device_id + layer_id) ----
psk = tables["layers"].loc[tables["layers"]["layer_type"] == "PSK",
                           ["device_id", "layer_id", "material"]].rename(columns={"material": "layer_material"})
j = psk.merge(tables["materials"], on=["device_id", "layer_id"], how="left")
j["prec"] = j["material"].map(canon_precursor)
j = j.dropna(subset=["prec"]).copy()          # เหลือเฉพาะแถวที่เป็นสารตั้งต้นที่รู้จัก
j = j[~j["prec"].isin(VOLATILE)]              # ตัดสารระเหย
j["ufam"] = j["unit"].map(unit_family)
j["amt"]  = pd.to_numeric(j["amount"], errors="coerce")

print(f"ชั้น PSK {len(psk)} แถว -> จับคู่สารตั้งต้นได้ {len(j)} แถว ({j['device_id'].nunique()} devices)")
display(j[["device_id", "prec", "amount", "unit", "ufam"]].head(8))

ชั้น PSK 95 แถว -> จับคู่สารตั้งต้นได้ 396 แถว (85 devices)

,device_id,prec,amount,unit,ufam
0,10.1002admi.202500231_Sample1,PbI2,324.0,mg,mass
1,10.1002admi.202500231_Sample1,CsI,11.0,mg,mass
2,10.1002admi.202500231_Sample1,PbBr2,47.0,mg,mass
3,10.1002admi.202500231_Sample1,MABr,14.0,NaN,NaN
4,10.1002admi.202500231_Sample1,FAI,110.0,mg,mass
5,10.1002admi.202500231_Sample2,PbI2,324.0,mg,mass
6,10.1002admi.202500231_Sample2,CsI,11.0,mg,mass
7,10.1002admi.202500231_Sample2,PbBr2,47.0,mg,mass


In [34]:
# ---- J4. คำนวณสัดส่วนจากสารตั้งต้น (ทีละ device) ----
prec_comp = {}     # device_id -> (A, X, b_site)
prec_status = {}   # device_id -> เหตุผลที่คำนวณได้/ไม่ได้
for dev, g in j.groupby("device_id"):
    fams = set(g["ufam"].dropna())
    bad  = g[g["ufam"].isna() | g["amt"].isna()]
    if len(bad):                                          # มีแถวแปลงไม่ได้ -> ไม่คำนวณเลย
        prec_status[dev] = "ตกหล่น: " + ", ".join(sorted(set(bad["prec"])))
    elif len(fams) != 1:                                  # หน่วยปนตระกูล -> สเกลเพี้ยน
        prec_status[dev] = f"หน่วยปนตระกูล {sorted(fams)}"
    elif not any(ION[p].get("B") for p in g["prec"]):      # ไม่มีแหล่ง Pb/Sn -> สูตรไม่ครบ
        prec_status[dev] = "ไม่มี B-site (สูตรไม่ครบ)"
    else:
        fam, A, X = fams.pop(), {}, {}
        conv = MOLAR if fam == "molar" else MASS
        for _, r in g.iterrows():
            mol = r["amt"] * conv[str(r["unit"]).strip().lower()]
            if fam == "mass":
                mol /= MW[r["prec"]]
            spec = ION[r["prec"]]
            for i, mlt in spec.get("A", {}).items(): A[i] = A.get(i, 0) + mol * mlt
            for i, mlt in spec.get("X", {}).items(): X[i] = X.get(i, 0) + mol * mlt
        sA, sX = sum(A.values()), sum(X.values())
        prec_comp[dev] = ({k: v/sA for k, v in A.items()}, {k: v/sX for k, v in X.items()}, "Pb")
        prec_status[dev] = f"ok ({fam})"

print(f"คำนวณจากสารตั้งต้นสำเร็จ {len(prec_comp)}/{len(prec_status)} devices")
display(pd.Series(prec_status).str.replace(r"[:(].*", "", regex=True).str.strip()
          .value_counts().rename("n_devices").to_frame())

คำนวณจากสารตั้งต้นสำเร็จ 46/85 devices

,n_devices
ok,46
ตกหล่น,39


In [35]:
_bad = {d: v for d, v in prec_status.items() if not v.startswith("ok")}
print(f"ตกหล่น {len(_bad)} devices จาก {len({d.rsplit('_Sample', 1)[0] for d in _bad})} paper")

_sum = (pd.DataFrame([{"paper": d.rsplit("_Sample", 1)[0], "เหตุผล": v} for d, v in _bad.items()])
          .groupby(["paper", "เหตุผล"]).size().rename("devices")
          .reset_index().sort_values("devices", ascending=False))
display(_sum)

print("แถวที่แปลงไม่ได้ (ต้นเหตุจริง):")
_p = j.loc[j["ufam"].isna() | j["amt"].isna(), ["device_id", "prec", "amount", "unit", "ufam"]]
display(_p.sort_values(["device_id", "prec"]))

ตกหล่น 39 devices จาก 13 paper


,paper,เหตุผล,devices
5,10.1038_s41598-023-32697-z,"ตกหล่น: CsI, FAI, MABr, PbI2",5
1,10.1016_j.solmat.2025.113858,"ตกหล่น: CsI, FAI, MABr, PbBr2, PbI2",4
0,10.1002admi.202500231,ตกหล่น: MABr,4
12,10_1039_d3lf00258f,"ตกหล่น: CsI, FAI, MABr, PbBr2, PbI2",4
3,10.1016j.nanoen.2021.106285,ตกหล่น: CsI,3
9,10_1021_acsaem_2c03072,"ตกหล่น: CsI, FAI, MAI, PbBr2",3
4,10.1021_acsenergylett.3c00697,"ตกหล่น: CsI, FAI, PbBr2, PbI2",3
11,10_1021_acsami_5c15868,ตกหล่น: CsI,3
2,10.1016j.jechem.2021.11.029,"ตกหล่น: FAI, MABr, PbBr2, PbI2",2
8,10_1016_j_solener_2025_113286,ตกหล่น: CsI,2


แถวที่แปลงไม่ได้ (ต้นเหตุจริง):


,device_id,prec,amount,unit,ufam
3,10.1002admi.202500231_Sample1,MABr,14.0,NaN,NaN
8,10.1002admi.202500231_Sample2,MABr,14.0,NaN,NaN
13,10.1002admi.202500231_Sample3,MABr,14.0,NaN,NaN
18,10.1002admi.202500231_Sample4,MABr,14.0,NaN,NaN
102,10.1016_j.solmat.2025.113858_Sample1,CsI,0.05,mol ratio,NaN
103,10.1016_j.solmat.2025.113858_Sample1,FAI,0.85,mol ratio,NaN
104,10.1016_j.solmat.2025.113858_Sample1,MABr,0.05,mol ratio,NaN
106,10.1016_j.solmat.2025.113858_Sample1,PbBr2,0.05,mol ratio,NaN
105,10.1016_j.solmat.2025.113858_Sample1,PbI2,0.95,mol ratio,NaN
109,10.1016_j.solmat.2025.113858_Sample2,CsI,0.05,mol ratio,NaN


In [36]:
# ---- J5. รวม 2 แหล่ง: สูตรเป็นหลัก -> precursor เติมช่องว่าง ----
TOL = 0.03      # เกณฑ์ยอมรับตอนเทียบ 2 แหล่ง
rows = []
for _, r in psk.iterrows():
    dev = r["device_id"]
    f_res, p_res = composition_from_formula(r["layer_material"]), prec_comp.get(dev)
    row = {"device_id": dev, "layer_material": r["layer_material"],
           "precursor_status": prec_status.get(dev, "ไม่มีแถว precursor")}
    if f_res:
        A, X, b = f_res
        row["comp_source"] = "formula"
        if p_res:      # มีทั้งคู่ -> ใช้สูตร แล้วเทียบ precursor เป็น cross-check
            pA, pX, _ = p_res
            diff = [k for k in set(A) | set(X) | set(pA) | set(pX)
                    if abs({**A, **X}.get(k, 0) - {**pA, **pX}.get(k, 0)) > TOL]
            row["cross_check"] = "ตรง" if not diff else "ต่าง: " + ", ".join(sorted(diff))
    elif p_res:
        A, X, b = p_res
        row["comp_source"] = "precursor"
        row["cross_check"] = "-- (สูตรแตกไม่ได้)"
    else:
        row["comp_source"] = "ไม่มีข้อมูล"
        rows.append(row)
        continue
    row["b_site"] = b
    row.update({f"a_{k}": round(v, 3) for k, v in A.items()})
    row.update({f"x_{k}": round(v, 3) for k, v in X.items()})
    rows.append(row)

In [37]:
psk_composition = pd.DataFrame(rows)
front = ["device_id", "comp_source", "cross_check", "b_site"]
psk_composition = psk_composition[[c for c in front if c in psk_composition.columns]
                                  + [c for c in psk_composition.columns if c not in front]]

n_dev = psk_composition["device_id"].nunique()
n_ok  = int((psk_composition["comp_source"] != "ไม่มีข้อมูล").sum())
print(f"ได้องค์ประกอบ {n_ok}/{n_dev} devices")
display(psk_composition["comp_source"].value_counts().rename("n_devices").to_frame())

ได้องค์ประกอบ 95/95 devices


,n_devices
comp_source,
formula,83
precursor,12


In [38]:
_diff = psk_composition[psk_composition["cross_check"].fillna("").str.startswith("ต่าง")]
if len(_diff):
    print("เคสที่ไม่ตรง — วิเคราะห์ว่าขาดสารตัวไหน:")
    for _, r in _diff.drop_duplicates(subset=["layer_material"]).iterrows():
        dev = r["device_id"]
        fa = {k[2:]: r[k] for k in psk_composition.columns if k.startswith("a_") and pd.notna(r[k])}
        fx = {k[2:]: r[k] for k in psk_composition.columns if k.startswith("x_") and pd.notna(r[k])}
        pA, pX, _b = prec_comp[dev]
        _g = j[j["device_id"] == dev]
        _have = set(_g["prec"])

        print(f"\n  {dev}")
        print(f"    สูตรนาม            : {r['layer_material']}")
        print(f"    formula (ใช้ค่านี้)  : A {fa}  X {fx}")
        print(f"    precursor คำนวณได้  : A { {k: round(v,3) for k,v in pA.items()} }  X { {k: round(v,3) for k,v in pX.items()} }")
        print(f"    สารตั้งต้นที่กรอกไว้  : {', '.join(f'{a} {b} {c}' for a,b,c in zip(_g['prec'], _g['amount'], _g['unit']))}")

        # ไอออนไหนที่ precursor ให้ได้ "น้อยกว่า" สูตร -> ขาดแหล่งของไอออนนั้น
        _short = {i: (w, {**pA, **pX}.get(i, 0.0))
                  for i, w in {**fa, **fx}.items() if w - {**pA, **pX}.get(i, 0.0) > TOL}
        if not _short:
            print("    -> ต่างแบบไม่ได้ขาด (สัดส่วนเพี้ยนจากตัวอื่นเกินมา)")
            continue
        for _ion, (_want, _got) in _short.items():
            _src = [p for p, spec in ION.items()
                    if _ion in {**spec.get("A", {}), **spec.get("B", {}), **spec.get("X", {})}]
            _absent = [p for p in _src if p not in _have]
            _msg = (f"ไม่มีในรายการ: {', '.join(_absent)}" if _absent
                    else f"มีครบ ({', '.join(_src)}) แต่ปริมาณน้อยไป")
            print(f"    -> ขาด {_ion}: สูตรต้องการ {_want:.3f} แต่สารที่กรอกให้ได้ {_got:.3f}  |  {_msg}")

        _over = [i for i, w in {**fa, **fx}.items() if {**pA, **pX}.get(i, 0) - w > TOL]
        if _over:
            print(f"    -> {', '.join(_over)} ดูเหมือนเกิน แต่เป็นผลจากตัวข้างบนขาด (สัดส่วนถูกดันขึ้น)")

เคสที่ไม่ตรง — วิเคราะห์ว่าขาดสารตัวไหน:



  10.1007_s10854-017-8094-9_Sample1
    สูตรนาม            : Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3
    formula (ใช้ค่านี้)  : A {'Cs': 0.08, 'FA': 0.764, 'MA': 0.156}  X {'I': 0.83, 'Br': 0.17}
    precursor คำนวณได้  : A {'FA': 0.37, 'MA': 0.074, 'Cs': 0.556}  X {'I': 0.887, 'Br': 0.113}
    สารตั้งต้นที่กรอกไว้  : PbI2 1.1 M, PbBr2 0.2 M, FAI 1.0 M, MABr 0.2 M, CsI 1.5 M
    -> ขาด FA: สูตรต้องการ 0.764 แต่สารที่กรอกให้ได้ 0.370  |  มีครบ (FAI) แต่ปริมาณน้อยไป
    -> ขาด MA: สูตรต้องการ 0.156 แต่สารที่กรอกให้ได้ 0.074  |  ไม่มีในรายการ: MAI, MACl
    -> ขาด Br: สูตรต้องการ 0.170 แต่สารที่กรอกให้ได้ 0.113  |  ไม่มีในรายการ: CsBr
    -> Cs, I ดูเหมือนเกิน แต่เป็นผลจากตัวข้างบนขาด (สัดส่วนถูกดันขึ้น)

  10.1016_j.joule.2022.06.026_Sample1
    สูตรนาม            : Cs0.1(FA0.87MA0.13)0.9Pb(I0.87Br0.13)3
    formula (ใช้ค่านี้)  : A {'Cs': 0.1, 'FA': 0.783, 'MA': 0.117}  X {'I': 0.87, 'Br': 0.13}
    precursor คำนวณได้  : A {'FA': 0.919, 'Cs': 0.081}  X {'I': 1.0}
    สารตั้งต้นที่กรอก

In [39]:
display(psk_composition.dropna(axis=1, how="all"))

,device_id,comp_source,cross_check,b_site,layer_material,precursor_status,a_Cs,a_FA,a_MA,x_I,x_Br,a_Rb
0,10.1002admi.202500231_Sample1,formula,NaN,Pb,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ตกหล่น: MABr,0.050,0.790,0.160,0.870,0.130,NaN
1,10.1002admi.202500231_Sample2,formula,NaN,Pb,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ตกหล่น: MABr,0.050,0.790,0.160,0.870,0.130,NaN
2,10.1002admi.202500231_Sample3,formula,NaN,Pb,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ตกหล่น: MABr,0.050,0.790,0.160,0.870,0.130,NaN
3,10.1002admi.202500231_Sample4,formula,NaN,Pb,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ตกหล่น: MABr,0.050,0.790,0.160,0.870,0.130,NaN
4,10.1007_s10854-017-8094-9_Sample1,formula,"ต่าง: Br, Cs, FA, I, MA",Pb,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,ok (molar),0.080,0.764,0.156,0.830,0.170,NaN
5,10.1007_s10854-017-8094-9_Sample2,formula,"ต่าง: Br, Cs, FA, I, MA",Pb,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,ok (molar),0.080,0.764,0.156,0.830,0.170,NaN
6,10.1007_s10854-017-8094-9_Sample3,formula,"ต่าง: Br, Cs, FA, I, MA",Pb,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,ok (molar),0.080,0.764,0.156,0.830,0.170,NaN
7,10.1007_s10854-017-8094-9_Sample4,formula,"ต่าง: Br, Cs, FA, I, MA",Pb,Cs0.08(MA0.17FA0.83)0.92Pb(I0.83Br0.17)3,ok (molar),0.080,0.764,0.156,0.830,0.170,NaN
8,10.1016.jpowsour.2020.228738_Sample1,precursor,-- (สูตรแตกไม่ได้),Pb,NaN,ok (mass),0.539,0.234,0.227,0.839,0.161,NaN
9,10.1016.jpowsour.2020.228738_Sample2,precursor,-- (สูตรแตกไม่ได้),Pb,PbI2 + PbBr2 + FAI + MABr + CsI,ok (mass),0.539,0.234,0.227,0.839,0.161,NaN


## I. `properties` unit ผิด — explore

2 ปัญหาคนละแบบในคอลัมน์ `unit`

**1. Voc/Jsc ติด unit `%`** (12 แถว, paper `10.1016j.jechem.2021.11.029`)
ค่า Voc 1.06-1.12 และ Jsc 20.7-22.1 อยู่ในช่วงปกติของ V และ mA cm-2 พอดี → **ป้ายผิด ค่าถูก**

**2. mobility หน่วย m²** (3 แถว, paper `10.1021_acsami.5c16972`)
paper อื่นในชุดใช้ `cm2 V-1 s-1` เป็นมาตรฐาน → **ต้องคูณ 10⁴**

In [40]:
def metric_of(name):
    n = str(name).lower()
    for k, b in [("voc", "Voc"), ("jsc", "Jsc"), ("mobility", "mobility")]:
        if k in n: return b
    return "other"

prop = tables["properties"]
metric = prop["property"].map(metric_of)
u = prop["unit"].astype(str).str.strip()

mislabel_idx = prop.index[(u == "%") & metric.isin(["Voc", "Jsc"])]
mobility_m2_idx = prop.index[(metric == "mobility") & u.str.startswith("m2")]

print(f"Voc/Jsc ติด unit '%'          : {len(mislabel_idx)} แถว")
print(f"mobility หน่วย m2 จริง (x1e4) : {len(mobility_m2_idx)} แถว "
      f"(audit เดิมนับผิดเป็น 7 — 4 ใน นั้นคือ 'cm2 V-1 s-1' ที่ถูกอยู่แล้ว ไม่ใช่ m2)")
prop.loc[list(mislabel_idx) + list(mobility_m2_idx), ["device_id", "property", "value", "unit"]]

Voc/Jsc ติด unit '%'          : 12 แถว
mobility หน่วย m2 จริง (x1e4) : 0 แถว (audit เดิมนับผิดเป็น 7 — 4 ใน นั้นคือ 'cm2 V-1 s-1' ที่ถูกอยู่แล้ว ไม่ใช่ m2)


,device_id,property,value,unit
223,10.1016j.jechem.2021.11.029_Sample1,Voc (RS),1.064,%
224,10.1016j.jechem.2021.11.029_Sample1,Jsc (RS),21.32,%
227,10.1016j.jechem.2021.11.029_Sample1,Voc (FS),1.061,%
228,10.1016j.jechem.2021.11.029_Sample1,Jsc (FS),21.12,%
231,10.1016j.jechem.2021.11.029_Sample1,Average Voc,1.065,%
232,10.1016j.jechem.2021.11.029_Sample1,Average Jsc,20.78,%
243,10.1016j.jechem.2021.11.029_Sample2,Voc (RS),1.118,%
244,10.1016j.jechem.2021.11.029_Sample2,Jsc (RS),21.98,%
247,10.1016j.jechem.2021.11.029_Sample2,Voc (FS),1.122,%
248,10.1016j.jechem.2021.11.029_Sample2,Jsc (FS),22.12,%


## I. `properties` unit ผิด — fix

**⚠️ บั๊กใน audit เดิม**: ใช้ `unit.str.contains("m2 V")` ซึ่งแมตช์ `"cm2 V-1 s-1"` ติดมาด้วย (มี `"m2 V"` เป็น substring) → นับผิดเป็น 7 แถว ทั้งที่ผิดจริงแค่ 3

ถ้า apply ตามนั้นจะทำให้ 4 แถวที่ถูกอยู่แล้วพังไปหมื่นเท่า (43.2 → 432000 cm²/Vs) แก้เป็น `startswith("m2")` แทน

**sanity check**: หลังคูณ 10⁴ ได้ 3.67 / 4.65 / 6.94 cm²/Vs อยู่ในช่วงเดียวกับ paper อื่น (3.46, 43.2) — สมเหตุผล

**สิ่งที่ทำ**: เพิ่ม 3 คอลัมน์ `value_after_unit_fix` / `unit_corrected` / `unit_fix_note` (ไม่แตะ `value`/`unit` เดิม ตามกติกา E-H)

| เคส | `value_after_unit_fix` | `unit_corrected` |
|---|---|---|
| Voc/Jsc `%` | เท่าเดิม | `V` / `mA cm-2` |
| mobility m² | ×10⁴ | `cm2 ...` |

In [41]:
VOC_JSC_UNIT = {"Voc": "V", "Jsc": "mA cm-2"}   # ตรงกับ unit ที่ถูกอยู่แล้วในตารางนี้

# audit ข้อ M: ชื่อเดิม "value_corrected" สื่อว่าเป็นค่าที่แก้แล้วทุกชนิด ซึ่งไม่จริง
#   คอลัมน์นี้เก็บเฉพาะการแก้ที่เกิดจากหน่วยผิด (m2 -> cm2) เท่านั้น
#   การแก้ชนิดอื่น (FF x100 ช่วง 3, Tauc Eg §8.8, เครื่องหมาย VBM §8.8b) ไปอยู่ที่ value_parsed
#   -> เปลี่ยนชื่อเป็น value_after_unit_fix ให้ตรงกับสิ่งที่มันเก็บจริง
prop["unit_corrected"] = prop["unit"]
_vf = prop["value"].map(to_num_sci)
prop["value_after_unit_fix"] = _vf.map(lambda t: t[0])
prop["unit_fix_note"] = _vf.map(lambda t: t[1])   # ว่าง = ไม่ต้องตีความ

for idx in mislabel_idx:
    m = metric.at[idx]
    correct_unit = VOC_JSC_UNIT[m]
    prop.at[idx, "unit_corrected"] = correct_unit
    prop.at[idx, "unit_fix_note"] = f"unit เดิม '%' ผิดชนิด แก้เป็น '{correct_unit}' (ไม่แตะค่า value)"

for idx in mobility_m2_idx:
    old_val = prop.at[idx, "value_after_unit_fix"]
    new_val = old_val * 1e4
    prop.at[idx, "value_after_unit_fix"] = new_val
    prop.at[idx, "unit_corrected"] = prop.at[idx, "unit"].replace("m2", "cm2", 1)
    prop.at[idx, "unit_fix_note"] = f"หน่วย m2 -> cm2 (x1e4): {old_val:g} -> {new_val:g}"

# แถวที่ต้องตีความสัญกรณ์ยกกำลังก็ต้องมีร่องรอยใน audit เหมือนการตีความอื่น ๆ ของช่วง 2
#   (ไม่งั้นด่านตรวจ "ทุกแถวที่ parse ตรง ๆ ไม่ได้ ต้องอยู่ใน audit" จะไม่ผ่าน)
for _i in prop.index[prop["unit_fix_note"].fillna("").str.startswith("สัญกรณ์ยกกำลัง")]:
    audit_rows.append({"table": "properties", "column": "value",
                       "device_id": prop.at[_i, "device_id"], "raw": prop.at[_i, "value"],
                       "parsed": prop.at[_i, "value_after_unit_fix"], "vmin": None, "vmax": None,
                       "extra": prop.at[_i, "property"], "note": prop.at[_i, "unit_fix_note"]})

n_fixed = len(mislabel_idx) + len(mobility_m2_idx)
print(f"แก้แล้ว {n_fixed} แถว ({len(mislabel_idx)} unit mislabel + {len(mobility_m2_idx)} mobility m2->cm2)")
prop.loc[list(mislabel_idx) + list(mobility_m2_idx),
         ["device_id", "property", "value", "unit", "value_after_unit_fix", "unit_corrected", "unit_fix_note"]]

แก้แล้ว 12 แถว (12 unit mislabel + 0 mobility m2->cm2)


,device_id,property,value,unit,value_after_unit_fix,unit_corrected,unit_fix_note
223,10.1016j.jechem.2021.11.029_Sample1,Voc (RS),1.064,%,1.064,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
224,10.1016j.jechem.2021.11.029_Sample1,Jsc (RS),21.32,%,21.320,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
227,10.1016j.jechem.2021.11.029_Sample1,Voc (FS),1.061,%,1.061,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
228,10.1016j.jechem.2021.11.029_Sample1,Jsc (FS),21.12,%,21.120,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
231,10.1016j.jechem.2021.11.029_Sample1,Average Voc,1.065,%,1.065,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
232,10.1016j.jechem.2021.11.029_Sample1,Average Jsc,20.78,%,20.780,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
243,10.1016j.jechem.2021.11.029_Sample2,Voc (RS),1.118,%,1.118,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
244,10.1016j.jechem.2021.11.029_Sample2,Jsc (RS),21.98,%,21.980,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
247,10.1016j.jechem.2021.11.029_Sample2,Voc (FS),1.122,%,1.122,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
248,10.1016j.jechem.2021.11.029_Sample2,Jsc (FS),22.12,%,22.120,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)


## K. `materials.solvent` — explore

`solvent` เขียนกันคนละแบบหนัก ทั้งที่ระบบตัวทำละลายมีไม่กี่ระบบ ตัวอย่างที่เป็นสิ่งเดียวกันทั้งหมด

`DMF/DMSO (4/1 V/V)` = `DMF:DMSO (4:1 v/v)` = `DMF/DMSO (8:2 V/V)` = `DMF/DMSO (800/200 µL)` = `DMSO/DMF (1:4 v/v)`

**ทำไมคุ้มแก้** — สัดส่วน DMF:DMSO เป็น process knob ที่ส่งผลต่อ grain size โดยตรง ถ้าไม่ normalize จะนับเป็นคนละกลุ่มทั้งที่เป็นสูตรเดียวกัน และใช้เป็น feature ตัวเลขไม่ได้

In [42]:
# ---- K. explore: ดูว่า solvent เขียนกี่แบบ ----
_s = tables["materials"]["solvent"].dropna().str.strip()
print(f"solvent ไม่ว่าง {len(_s)} แถว | เขียน {_s.nunique()} แบบ")
display(_s.value_counts().rename("n_rows").to_frame())

solvent ไม่ว่าง 756 แถว | เขียน 43 แบบ


,n_rows
solvent,
DMF/DMSO (4:1 v/v),243
IPA,80
DMSO:DMF (1:4 v/v),50
DMF:DMSO (1:4 v/v),30
Chlorobenzene,27
CB,25
Ethanol,25
H2O,20
DMF:DMSO (8:2 v/v),20


## K. `materials.solvent` — fix

แตกเป็น 4 คอลัมน์ใหม่ (ไม่ทับ `solvent` เดิม ตามกติกาช่วง 2)

| คอลัมน์ | ความหมาย |
|---|---|
| `solvent_1`, `solvent_2` | ชื่อมาตรฐาน **เรียงตามตัวอักษรเสมอ** — `DMSO/DMF` จึงกลายเป็น `DMF`,`DMSO` พร้อมกลับอัตราส่วนให้ |
| `solvent_ratio` | อัตราส่วน `solvent_1 : solvent_2` เป็นตัวเลข (`800/200` → 4.0, `8:2` → 4.0) |
| `solvent_system` | สตริงมาตรฐานไว้ group เช่น `DMF:DMSO (4:1)` (ย่อเศษส่วนด้วย gcd) |
| `solvent_note` | บันทึกกรณีตีความไม่ครบ เช่น วงเล็บเป็นความเข้มข้นไม่ใช่อัตราส่วน |

ชื่อพ้องที่รวมให้ด้วย: `CB` = `Chlorobenzene` / `H2O` = `Deionized water` = `De-ionized water` / `ethanol` = `Anhydrous ethanol` = `ethyl alcohol` / `IPA` = `Anhydrous IPA`

In [43]:
# ---- K. fix: normalize ชื่อ + อัตราส่วน ----
from math import gcd

SOLVENT_CANON = {
    "dmf": "DMF", "dmso": "DMSO",
    "cb": "Chlorobenzene", "chlorobenzene": "Chlorobenzene",
    "ipa": "IPA", "anhydrousipa": "IPA", "isopropanol": "IPA",
    "ethanol": "Ethanol", "anhydrousethanol": "Ethanol",
    "ethylalcohol": "Ethanol", "etoh": "Ethanol",
    "methanol": "Methanol",
    "h2o": "Water", "water": "Water", "deionizedwater": "Water",
    "deionisedwater": "Water", "diwater": "Water",
    "acetonitrile": "Acetonitrile",
    "ethyleneglycol": "Ethylene glycol",
    "ethanolamine": "Ethanolamine",
    "toluene": "Toluene", "ethylacetate": "Ethyl acetate",
}
NOISE_WORDS = {"stock", "solution", "anhydrous", "solvent"}
def canon_solvent(tok):
    k = re.sub(r"[^a-z0-9]", "", str(tok).lower())
    k = re.sub(r"^(de|di)ionized", "deionized", k)
    if k in SOLVENT_CANON:
        return SOLVENT_CANON[k]
    # เผื่อมีคำเกิน เช่น "DMF stock" -> ลองทีละคำ
    for w in re.split(r"\s+", str(tok).strip()):
        kw = re.sub(r"[^a-z0-9]", "", w.lower())
        if kw in NOISE_WORDS:
            continue
        if kw in SOLVENT_CANON:
            return SOLVENT_CANON[kw]
    return None

def parse_solvent(v):
    """คืน dict: solvent_1, solvent_2, solvent_ratio (s1:s2), solvent_system, solvent_note"""
    out = {"solvent_1": None, "solvent_2": None, "solvent_ratio": None,
           "solvent_system": None, "solvent_note": ""}
    if not isinstance(v, str) or not v.strip():
        return out
    s = v.strip()

    # แยกส่วนอัตราส่วนในวงเล็บออกจากส่วนชื่อ
    ratio_txt, head = None, s
    m = re.search(r"\(([^)]*)\)", s)
    if m:
        head, ratio_txt = s[:m.start()].strip(), m.group(1)

    # ชื่อตัวทำละลาย: คั่นด้วย / : , หรือติดกัน (DMFDMSO)
    toks = [t for t in re.split(r"[/:,]| and ", head) if t.strip()]
    names = [canon_solvent(t) for t in toks]
    if len(names) == 1 and names[0] is None:                 # เผื่อเขียนติดกัน เช่น DMFDMSO
        k = re.sub(r"[^a-z0-9]", "", head.lower())
        for a in ("dmf", "dmso"):
            for b in ("dmf", "dmso"):
                if a != b and k == a + b:
                    names = [SOLVENT_CANON[a], SOLVENT_CANON[b]]
    names = [n for n in names if n]
    if not names:
        out["solvent_note"] = f"จำแนกชื่อไม่ได้: {s!r}"
        return out

    # อัตราส่วน: 4:1 / 4/1 / 800/200 -> ตัวเลข 2 ตัวแรก (หน่วยไม่สำคัญ เพราะใช้เป็นสัดส่วน)
    r1 = r2 = None
    if ratio_txt:
        nums = re.findall(r"\d+(?:\.\d+)?", ratio_txt)
        if len(nums) >= 2 and not re.search(r"mg|mol|wt", ratio_txt, re.I):
            r1, r2 = float(nums[0]), float(nums[1])
        elif nums:
            out["solvent_note"] = f"วงเล็บไม่ใช่อัตราส่วน: ({ratio_txt})"

    if len(names) == 1:
        out.update(solvent_1=names[0], solvent_system=names[0])
        return out

    s1, s2 = names[0], names[1]
    # จัดลำดับให้เป็นมาตรฐาน (เรียงตามตัวอักษร) -> DMSO/DMF (1:4) กับ DMF/DMSO (4:1) จะรวมกลุ่มกันได้
    if s1 > s2:
        s1, s2 = s2, s1
        r1, r2 = (r2, r1) if r1 is not None else (None, None)
    out["solvent_1"], out["solvent_2"] = s1, s2
    if r1 and r2:
        out["solvent_ratio"] = round(r1 / r2, 3)
        a, b = r1, r2
        if float(a).is_integer() and float(b).is_integer():   # ย่อเศษส่วน 800:200 -> 4:1, 8:2 -> 4:1
            gg = gcd(int(a), int(b)); a, b = int(a)//gg, int(b)//gg
        else:
            a, b = round(r1/r2, 2), 1
        out["solvent_system"] = f"{s1}:{s2} ({a}:{b})"
    else:
        out["solvent_system"] = f"{s1}:{s2}"
    return out

In [44]:
# ไม่ทับคอลัมน์เดิม -- เพิ่มคอลัมน์ใหม่ตามกติกาช่วง 2 (`solvent` เดิมยังอยู่ครบ)
mat = tables["materials"]
_sv = pd.DataFrame([parse_solvent(v) for v in mat["solvent"]], index=mat.index)
for c in ["solvent_1", "solvent_2", "solvent_ratio", "solvent_system", "solvent_note"]:
    mat[c] = _sv[c]

_nn = mat[mat["solvent"].notna()]
print(f"normalize solvent: {_nn['solvent'].nunique()} แบบการเขียน -> {_nn['solvent_system'].nunique()} ระบบตัวทำละลาย")

normalize solvent: 43 แบบการเขียน -> 22 ระบบตัวทำละลาย


In [45]:
_before = tables["materials"]["solvent"].dropna().str.strip()
_after = _nn["solvent_system"]
print(f"ก่อนแก้ : {len(_before)} แถว | เขียน {_before.nunique()} แบบ")
print(f"หลังแก้ : {len(_after)} แถว | เหลือ {_after.nunique()} ระบบตัวทำละลาย")
display(_after.value_counts().rename("n_rows").to_frame())

ก่อนแก้ : 756 แถว | เขียน 43 แบบ


หลังแก้ : 756 แถว | เหลือ 22 ระบบตัวทำละลาย


,n_rows
solvent_system,
DMF:DMSO (4:1),341
IPA,89
Chlorobenzene,52
Ethanol,49
Water,42
DMF:DMSO (1:4),30
DMF,30
DMSO,15
Ethylene glycol,15


In [46]:
print("before -> after (เฉพาะที่รวมกลุ่มได้มากกว่า 1 แบบ):")
_chk = (_nn.groupby("solvent_system")["solvent"]
           .agg(n_rows="size", n_spellings="nunique",
                spellings=lambda s: "  |  ".join(f"{v} x{c}" for v, c in s.value_counts().items())))
display(_chk[_chk["n_spellings"] > 1].sort_values("n_spellings", ascending=False))

_bad = _nn[_nn["solvent_note"] != ""]
if len(_bad):
    print(f"แถวที่ parse ไม่สมบูรณ์ {len(_bad)} แถว — ต้นทางที่ต้องไปแก้:")
    for _i, _r in _bad.iterrows():
        _dev = mat.at[_i, "device_id"]
        print(f"\n  row {_i}  |  {_dev}")
        print(f"    ไฟล์ต้นทาง : Raw Data/{_dev}.xlsx  ->  sheet 'Materials'  ->  LayerID {mat.at[_i, 'layer_id']}")
        print(f"    ค่าที่กรอก  : material={mat.at[_i, 'material']!r}  type={mat.at[_i, 'type']!r}  "
              f"solvent={_r['solvent']!r}  amount={mat.at[_i, 'amount']!r}  unit={mat.at[_i, 'unit']!r}")
        print(f"    parse ได้   : solvent_1={_r['solvent_1']!r}  solvent_2={_r['solvent_2']!r}  ratio={_r['solvent_ratio']!r}")
        print(f"    ปัญหา      : {_r['solvent_note']}")

before -> after (เฉพาะที่รวมกลุ่มได้มากกว่า 1 แบบ):


,n_rows,n_spellings,spellings
solvent_system,,,
DMF:DMSO (4:1),341,5,DMF/DMSO (4:1 v/v) x243 | DMSO:DMF (1:4 v/v) x50 | DMF:DMSO (8:2 v/v) x20 | DMF/DMSO (4:1) x18 | DMF/DMSO (8...
DMF,30,4,DMF:NMP (9:1 v/v) x18 | DMF:2ME:NMP (0.45:0.48:0.07) x10 | DMF x1 | DMF stock (15 mg/mL) x1
Water,42,4,H2O x20 | Deionized water x13 | DI water x7 | Ammonia water x2
Ethanol,49,4,Ethanol x25 | Anhydrous ethanol x19 | Anhydrous EtOH x3 | Ethyl Alcohol x2
Acetonitrile,3,2,Acetonitrile + Ethanol x2 | Acetonitrile x1
Chlorobenzene,52,2,Chlorobenzene x27 | CB x25
IPA,89,2,IPA x80 | Anhydrous IPA x9
IPA:Water (1:3),7,2,DI Water:IPA (3:1 v/v) x5 | Water/IPA (3:1) x2


แถวที่ parse ไม่สมบูรณ์ 18 แถว — ต้นทางที่ต้องไปแก้:

  row 109  |  10.1016.jpowsour.2020.228738_Sample1
    ไฟล์ต้นทาง : Raw Data/10.1016.jpowsour.2020.228738_Sample1.xlsx  ->  sheet 'Materials'  ->  LayerID 6
    ค่าที่กรอก  : material='PCBM'  type='Solution'  solvent='DCB (anhydrous)'  amount='20.0'  unit='mg mL-1'
    parse ได้   : solvent_1=nan  solvent_2=nan  ratio=nan
    ปัญหา      : จำแนกชื่อไม่ได้: 'DCB (anhydrous)'

  row 110  |  10.1016.jpowsour.2020.228738_Sample1
    ไฟล์ต้นทาง : Raw Data/10.1016.jpowsour.2020.228738_Sample1.xlsx  ->  sheet 'Materials'  ->  LayerID 7
    ค่าที่กรอก  : material='ZnO NPs'  type='Nanoparticles'  solvent='2-propanol'  amount='2.5'  unit=nan
    parse ได้   : solvent_1=nan  solvent_2=nan  ratio=nan
    ปัญหา      : จำแนกชื่อไม่ได้: '2-propanol'

  row 121  |  10.1016.jpowsour.2020.228738_Sample2
    ไฟล์ต้นทาง : Raw Data/10.1016.jpowsour.2020.228738_Sample2.xlsx  ->  sheet 'Materials'  ->  LayerID 6
    ค่าที่กรอก  : material='PCBM'  type='Sol

In [47]:
_cols = ["solvent", "solvent_1", "solvent_2", "solvent_ratio", "solvent_system", "solvent_note"]
_map = (_nn[_cols].drop_duplicates(subset=["solvent"])
        .assign(n_rows=lambda d: d["solvent"].map(_nn["solvent"].value_counts()))
        .sort_values(["solvent_system", "solvent"])
        .reset_index(drop=True))
print(f"before -> after รายค่า ({len(_map)} แบบการเขียน -> {_nn['solvent_system'].nunique()} ระบบ)")
display(_map)

before -> after รายค่า (43 แบบการเขียน -> 22 ระบบ)


,solvent,solvent_1,solvent_2,solvent_ratio,solvent_system,solvent_note,n_rows
0,Acetonitrile,Acetonitrile,NaN,NaN,Acetonitrile,,1
1,Acetonitrile + Ethanol,Acetonitrile,NaN,NaN,Acetonitrile,,2
2,CB,Chlorobenzene,NaN,NaN,Chlorobenzene,,25
3,Chlorobenzene,Chlorobenzene,NaN,NaN,Chlorobenzene,,27
4,DMF,DMF,NaN,NaN,DMF,,1
5,DMF stock (15 mg/mL),DMF,NaN,NaN,DMF,วงเล็บไม่ใช่อัตราส่วน: (15 mg/mL),1
6,DMF:2ME:NMP (0.45:0.48:0.07),DMF,NaN,NaN,DMF,,10
7,DMF:NMP (9:1 v/v),DMF,NaN,NaN,DMF,,18
8,DMF:DMSO (1:4 v/v),DMF,DMSO,0.250,DMF:DMSO (1:4),,30
9,DMF/DMSO (2:1 v/v),DMF,DMSO,2.000,DMF:DMSO (2:1),,10


## L. `stability` เพิ่มเติม — explore

ต่อจากหัวข้อ F (ที่ parse `temperature_c` / `humidity_pct` / `retention_pct` ไปแล้ว) เหลืออีก 3 เรื่องที่ยังไม่ได้จัดการ

**1. `t80_hr` ว่างทั้งคอลัมน์** — T80 เป็นมาตรวัดที่เทียบข้าม paper ได้ดีที่สุด แต่ไม่มีสักแถว
และ**คำนวณเองไม่ได้** เพราะแต่ละอุปกรณ์มีจุดวัดจุดเดียว ไม่มีเส้นโค้งเวลา
(ประมาณเชิงเส้น `T80 ≈ t x 20/(100-R)` ไม่ควรทำ — perovskite เสื่อมแบบ burn-in เร็วแล้วช้า ไม่ใช่เส้นตรง)
→ **ยอมทิ้ง** ใช้ `retention_pct` คู่กับเงื่อนไขทดสอบเป็น covariate แทน

**2. `environment` เขียนกระจัดกระจาย** — 22 แบบ (`N2`, `N2 atmosphere`, `Nitrogen (N2)`, `ISOS-L-3`, `MPP`, `ambient`, ...)
ทั้งที่จริงยุบเป็นชนิดของ stress ได้ไม่กี่กลุ่ม

**3. `retention_pct` ปนสเกล + มีแถวที่คำนวณเองได้**
- 3 แถวค่า ≤ 1 (`0.4`, `0.8`, `0.97`) — **2 ใน 3 พิสูจน์ได้ว่าเป็น Excel percent format**
  (`number_format='0%'` ในไฟล์ `adma_202507730` Sample1/Sample5 — เก็บ `0.97` แต่หน้าจอโชว์ `97%`
  openpyxl อ่านค่าดิบ ตอน ETL จึงได้ `0.97`) ส่วนอีก 1 แถว (`heliyon` Sample3) พิมพ์ค่าดิบไว้เฉย ๆ ยังต้องเดา
  — สแกนทุกไฟล์แล้ว มีเฉพาะ 2 เซลล์นี้ที่ใช้ percent format ไม่มีคอลัมน์อื่นโดนผลนี้
- 1 แถวค่า > 100 (`100.85`) — เป็นไปได้จริงทางฟิสิกส์ (light-soaking) ไม่ใช่บั๊ก
- 4 แถวมี `initial_pce` + `final_pce` แต่ `retention_pct` ว่าง → คำนวณได้

In [48]:
# ---- L. explore ----
stb = tables["stability"]
_r = pd.to_numeric(stb["retention_pct_parsed"], errors="coerce")

print(f"t80_hr มีค่า        : {int(stb['t80_hr'].notna().sum())} แถว  <- ว่างทั้งคอลัมน์ ใช้ไม่ได้")
print(f"retention มีค่า     : {int(_r.notna().sum())} แถว | {stb.loc[_r.notna(), 'device_id'].nunique()} devices")
print(f"  ค่า <= 1 (สัดส่วน) : {int((_r <= 1).sum())} แถว")
print(f"  ค่า > 100         : {int((_r > 100).sum())} แถว")

_calc = _r.isna() & stb["initial_pce"].notna() & stb["final_pce"].notna()
print(f"  ว่างแต่มี initial+final pce (คำนวณได้): {int(_calc.sum())} แถว")

print(f"\nenvironment: {stb['environment'].nunique()} แบบ")
display(stb["environment"].value_counts().rename("n_rows").to_frame())

t80_hr มีค่า        : 2 แถว  <- ว่างทั้งคอลัมน์ ใช้ไม่ได้


retention มีค่า     : 74 แถว | 46 devices
  ค่า <= 1 (สัดส่วน) : 3 แถว
  ค่า > 100         : 1 แถว
  ว่างแต่มี initial+final pce (คำนวณได้): 4 แถว

environment: 29 แบบ


,n_rows
environment,
Ambient air,11
N2,9
Ambient,8
Air,6
Thermal Aging,5
N2-filled glovebox,5
ISOS-L2,4
N2/ISOS-L-2I,4
Nitrogen (N2),3


## L. `stability` เพิ่มเติม — fix

| ทำอะไร | ผล |
|---|---|
| เติม `retention` จาก `final_pce / initial_pce x100` | +4 แถว (24 → 28 devices) |
| ค่า ≤ 1 คูณ 100 | 3 แถว — **ติดธงว่าเป็นการอนุมาน** ยังไม่ยืนยันจาก paper |
| ค่า > 100 | ไม่แก้ ติดธงไว้เฉย ๆ |
| `environment` → `stress_group` | 22 แบบ → 6 กลุ่ม |

**`stress_group` ไม่ได้แก้ปัญหาเทียบกันไม่ได้** — แค่ลด cardinality เวลาใช้ต้องพก `temperature_c_parsed`
กับ `time_hr` เป็น covariate ไปด้วยเสมอ ห้ามใช้ `retention_pct` เดี่ยว ๆ

**หมายเหตุ**: `retention_pct` เดิมไม่ถูกแตะ — แก้ที่ `retention_pct_parsed` (คอลัมน์ของช่วง 2) ตามกติกา

In [49]:
# ---- L. fix ----
STRESS_RULES = [
    (r"isos-l|mpp|illumin|light|1-sun",  "light/operational"),
    (r"isos-t|thermal|damp heat",        "thermal"),
    (r"isos-d|storage|dark",             "dark storage"),
    (r"n2|nitrogen|glove",               "inert (N2)"),
    (r"air|ambient",                     "ambient air"),
]

def stress_group(v):
    """ยุบ environment ที่เขียนหลายแบบ เป็นชนิดของ stress"""
    if not isinstance(v, str) or not v.strip():
        return None
    t = v.lower()
    for pat, name in STRESS_RULES:
        if re.search(pat, t):
            return name
    return "other"

stb = tables["stability"]
stb["stress_group"] = stb["environment"].map(stress_group)

_before_dev = stb.loc[stb["retention_pct_parsed"].notna(), "device_id"].nunique()

# tables โหลดด้วย dtype=str -> initial_pce/final_pce ยังเป็นสตริง ต้องแปลงก่อนหาร
_ini = pd.to_numeric(stb["initial_pce"], errors="coerce")
_fin = pd.to_numeric(stb["final_pce"], errors="coerce")

# 1) เติมจาก initial/final PCE ก่อน (เฉพาะแถวที่ยังว่าง)
_m_calc = stb["retention_pct_parsed"].isna() & _ini.notna() & _fin.notna() & _ini.ne(0)
stb.loc[_m_calc, "retention_pct_parsed"] = (_fin[_m_calc] / _ini[_m_calc] * 100).round(2)
stb.loc[_m_calc, "retention_pct_note"] = "คำนวณจาก final_pce/initial_pce x100"

# 2) สเกลสัดส่วน -> %
#    ต้นเหตุ: ต้นฉบับ .xlsx จัดรูปแบบเซลล์เป็น Excel percent (number_format='0%')
#    -> ในไฟล์เก็บค่า 0.97 แต่หน้าจอโชว์ "97%" ; openpyxl อ่านค่าดิบมาเลย ตอน ETL จึงได้ 0.97
#    (สแกนทุกไฟล์แล้ว มีเฉพาะ 2 เซลล์นี้ที่ใช้ percent format -- adma_202507730 Sample1/Sample5)
XLSX_PERCENT_FMT = {"10_1002adma_202507730_Sample1", "10_1002adma_202507730_Sample5"}

stb["retention_pct_parsed"] = pd.to_numeric(stb["retention_pct_parsed"], errors="coerce")
_m_frac = stb["retention_pct_parsed"].notna() & (stb["retention_pct_parsed"] <= 1)
stb.loc[_m_frac, "retention_pct_parsed"] *= 100

_confirmed = _m_frac & stb["device_id"].isin(XLSX_PERCENT_FMT)
stb.loc[_confirmed, "retention_pct_note"] = "สเกล: x100 [ยืนยันแล้ว: ต้นฉบับใช้ Excel percent format '0%']"
stb.loc[_m_frac & ~_confirmed, "retention_pct_note"] = (
    "สเกล: x100 [อนุมาน -- ต้นฉบับพิมพ์ค่าดิบไว้ ไม่ได้จัดรูปแบบ percent ต้องเช็ค paper]")

# 3) > 100 ไม่แก้ค่า ติดธงอย่างเดียว
_m_over = stb["retention_pct_parsed"] > 100
stb.loc[_m_over, "retention_pct_note"] = "> 100% (เป็นไปได้จาก light-soaking) เก็บค่าเดิมไว้"

_after_dev = stb.loc[stb["retention_pct_parsed"].notna(), "device_id"].nunique()
print(f"retention ใช้ได้: {_before_dev} -> {_after_dev} devices")
print(f"environment {stb['environment'].nunique()} แบบ -> stress_group {stb['stress_group'].nunique()} กลุ่ม")

print("\nแถวที่ถูกแตะ:")
display(stb.loc[_m_calc | _m_frac | _m_over,
                ["device_id", "environment", "stress_group", "time_hr", "retention_pct",
                 "initial_pce", "final_pce", "retention_pct_parsed", "retention_pct_note"]])

def _rng(s):
    v = pd.to_numeric(s, errors="coerce").dropna()      # dtype=str -> ต้องแปลงก่อนหา min/max
    return f"{v.min():.0f}-{v.max():.0f}" if len(v) else "-"

print("แต่ละกลุ่มเทียบกันได้แค่ไหน (ยิ่งช่วงกว้าง ยิ่งเทียบตรง ๆ ไม่ได้):")
display(stb.dropna(subset=["retention_pct_parsed"]).groupby("stress_group").agg(
    n_devices=("device_id", "nunique"),
    temp_c=("temperature_c_parsed", _rng),
    time_hr=("time_hr", _rng),
    retention=("retention_pct_parsed", _rng)))

retention ใช้ได้: 46 -> 50 devices
environment 29 แบบ -> stress_group 6 กลุ่ม

แถวที่ถูกแตะ:


,device_id,environment,stress_group,time_hr,retention_pct,initial_pce,final_pce,retention_pct_parsed,retention_pct_note
0,10.1002admi.202500231_Sample1,ISOS-L2,light/operational,72,NaN,16.31,14.5,88.90,คำนวณจาก final_pce/initial_pce x100
1,10.1002admi.202500231_Sample2,ISOS-L2,light/operational,72,NaN,17.11,10.5,61.37,คำนวณจาก final_pce/initial_pce x100
2,10.1002admi.202500231_Sample3,ISOS-L2,light/operational,72,NaN,16.97,11.8,69.53,คำนวณจาก final_pce/initial_pce x100
3,10.1002admi.202500231_Sample4,ISOS-L2,light/operational,72,NaN,16.25,13.4,82.46,คำนวณจาก final_pce/initial_pce x100
52,10_1002_solr_202400534_Sample2,N2 storage,dark storage,1440,100.85,NaN,NaN,100.85,> 100% (เป็นไปได้จาก light-soaking) เก็บค่าเดิมไว้
55,10_1002adma_202507730_Sample1,ISOS-L-3,light/operational,580,0.4,NaN,NaN,40.00,สเกล: x100 [ยืนยันแล้ว: ต้นฉบับใช้ Excel percent format '0%']
56,10_1002adma_202507730_Sample5,ISOS-L-3 (MPPT),light/operational,1500,0.97,NaN,NaN,97.00,สเกล: x100 [ยืนยันแล้ว: ต้นฉบับใช้ Excel percent format '0%']
58,10_1016_j_heliyon_2024_e25352_Sample3,"N2 glovebox, AM1.5G 1-sun illumination",light/operational,144,0.8,NaN,NaN,80.00,สเกล: x100 [อนุมาน -- ต้นฉบับพิมพ์ค่าดิบไว้ ไม่ได้จัดรูปแบบ percent ต้องเช็ค paper]


แต่ละกลุ่มเทียบกันได้แค่ไหน (ยิ่งช่วงกว้าง ยิ่งเทียบตรง ๆ ไม่ได้):


,n_devices,temp_c,time_hr,retention
stress_group,,,,
ambient air,21,25-25,72-5000,48-95
dark storage,4,25-65,1000-1440,59-101
inert (N2),19,25-85,120-2880,20-95
light/operational,23,25-85,1-1500,40-97
thermal,4,85-85,400-2000,58-90


## สรุป + save audit + verify (จบช่วง 2 — ยังไม่เขียนดิสก์ ต่อด้วยช่วง 3 ก่อน)

In [50]:
audit = pd.DataFrame(audit_rows,
                     columns=["table", "column", "device_id", "raw",
                              "parsed", "vmin", "vmax", "extra", "note"])

ba = (audit.groupby(["table", "column", "raw", "parsed", "vmin", "vmax", "extra"],
                    dropna=False)
           .size().rename("n_rows").reset_index()
           .rename(columns={"raw": "before", "parsed": "after",
                            "extra": "kind/flag"})
           .sort_values(["table", "column", "before"])
           .reset_index(drop=True))
print(f"คู่ before -> after ไม่ซ้ำ: {len(ba)} | ครอบคลุม {ba.n_rows.sum()} แถวที่ถูกตีความ")
ba

คู่ before -> after ไม่ซ้ำ: 49 | ครอบคลุม 135 แถวที่ถูกตีความ


,table,column,before,after,vmin,vmax,kind/flag,n_rows
0,layers,thickness_nm,1/25,26.00000,1.0,25.0,ETL,4
1,layers,thickness_nm,1200 Å,120.00000,120.0,120.0,Cathode,2
2,layers,thickness_nm,280-320,300.00000,280.0,320.0,PSK,2
3,layers,thickness_nm,"4, 100",104.00000,4.0,100.0,Cathode,2
4,layers,thickness_nm,40-60,50.00000,40.0,60.0,HTL,2
5,layers,thickness_nm,400 Å,40.00000,40.0,40.0,ETL treatment (1),2
6,layers,thickness_nm,400-450,425.00000,400.0,450.0,PSK,4
7,layers,thickness_nm,90 Å,9.00000,9.0,9.0,ETL treatment (2),2
8,materials,amount,1:50,NaN,NaN,NaN,v:v,2
9,properties,value,1.01×10-3,0.00101,NaN,NaN,Conductivity,1


In [51]:
audit.to_csv(AUDIT / "value_parse_audit.csv", index=False)
print("บันทึก audit แล้ว:", AUDIT / "value_parse_audit.csv", f"({len(audit)} แถว)")
audit.groupby(["table", "column"]).size().rename("n_rows").to_frame()

บันทึก audit แล้ว: D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\value_parse_audit.csv (135 แถว)


n_rows
table      column               
layers     thickness_nm       20
materials  amount              2
properties value               2
stability  humidity_pct       29
           retention_pct       2
           t80_hr              1
           temperature_c      10
vendors    purity             69

In [52]:
raw_ok = all(tables[n][c].fillna("").equals(tables_after_text[n][c].fillna(""))
             for n in tables_after_text for c in tables_after_text[n].columns)
rows_ok = all(len(tables[n]) == len(tables_after_text[n]) for n in tables)

n_bad_total = 0
unhandled = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].fillna("").str.strip()
        bad_idx = s[(s != "") & ~s.map(is_num)].index
        n_bad_total += len(bad_idx)
        for i in bad_idx:
            note_cols = [c for c in tables[name].columns
                         if c.startswith(col) and c != col]
            vals = [str(tables[name].at[i, c]) for c in note_cols]
            handled = any(v not in ("", "nan", "None") for v in vals)
            flagged_bad = any("parse ไม่ได้" in v or v == "unrecognized" for v in vals)
            if not handled or flagged_bad:
                unhandled.append((name, col, int(i), s[i]))

print("1) คอลัมน์เดิมไม่ถูกแก้ระหว่างช่วง 2 :", raw_ok)
print("2) จำนวนแถวเท่าเดิมทุกตาราง            :", rows_ok)
print(f"3) เคส non-numeric {n_bad_total} แถว | ใน audit {len(audit)} แถว | หลุด: {len(unhandled)}")
if unhandled:
    # ต้องบอกให้ได้ว่าแถวไหน ไม่งั้นพอ Raw Data ชุดใหม่เข้ามาจะไล่หาไม่เจอ
    _u = pd.DataFrame(unhandled, columns=["ตาราง", "คอลัมน์", "row", "ค่า"])
    print()
    print(f"แถวที่ยังไม่มีใครตีความ ({len(_u)}) -- ต้องเพิ่มกฎ parser หรือยืนยันว่าปล่อยว่างได้:")
    display(_u.groupby(["ตาราง", "คอลัมน์"])
              .agg(n=("ค่า", "size"), ตัวอย่าง=("ค่า", lambda x: " | ".join(sorted(set(x))[:6]))))
assert raw_ok and rows_ok and n_bad_total == len(audit) and not unhandled, "VERIFY FAILED"
print("ช่วง 2 (value-format) verify ผ่านหมด ✓")

1) คอลัมน์เดิมไม่ถูกแก้ระหว่างช่วง 2 :

 True
2) จำนวนแถวเท่าเดิมทุกตาราง            : True
3) เคส non-numeric 135 แถว | ใน audit 135 แถว | หลุด: 0
ช่วง 2 (value-format) verify ผ่านหมด ✓


# ช่วง 3 — Prepare properties (แตกชื่อ + แปลงสเกล FF) → `processed/prepared/`

## 3.0 ทำไมต้องแตกชื่อ property

ตอนนี้ `properties.property` มี 91 ชื่อ ซึ่งจริง ๆ เป็นการเอา **5 มิติ** มาต่อกันเป็น string เดียว:

```
"Champion PCE (FS)"  =  base: PCE       + stat: champion + scan: FS
"HTL grain size"     =  base: Grain size + layer: HTL
"Hole VTFL"          =  base: VTFL      + carrier: hole
```

ถ้าไม่แตก มิติพวกนี้จะฝังอยู่ในชื่อ ทำให้ pivot/slice ไม่ได้ (เช่น "เอาเฉพาะ champion + reverse scan")
และ 91 ชื่อจะกลายเป็น 91 features กระจัดกระจายตอนทำ ML — แตกแล้วเหลือ **base 46 ตัว** + คอลัมน์กรอง 4 ตัว

| มิติ | รูปแบบในชื่อ | ค่า | หมายเหตุ |
|---|---|---|---|
| `stat` | คำนำหน้า Champion/Average/Avg/Certified/Verified | เก็บ**ตามที่เขียน** (lower-case) | ⚠️ `avg` ≠ `average`, `verified` ≠ `certified` — **ตั้งใจไม่รวม** เพราะเป็นกลุ่มรอยืนยัน มี `STAT_MERGE_PENDING` เตรียมไว้ เปิดใช้เมื่อยืนยันแล้ว |
| `scan` | วงเล็บท้าย (RS)/(FS) | RS / FS | reverse / forward scan ของ J-V |
| `layer` | คำนำหน้า HTL/PSK หรือวงเล็บท้าย (HTL)/(PVK) | HTL / PSK | `PVK→PSK` เป็นแค่ตัวสะกด (ไม่ใช่ semantic) |
| `carrier` | คำนำหน้า Electron/Hole | electron / hole | ใช้กับ VTFL, trap density, mobility |

หลังตัด prefix/suffix แล้ว base ที่เหลือบางตัวสะกดต่างกันแค่ตัวพิมพ์ → ใช้กติกา **variant-merge แบบเดียวกับช่วง 1** (majority vote, case/วรรคตอนเท่านั้น) ปิดท้าย

## 3.0b กติกาแปลงสเกล FF

FF ถูกรายงาน 2 สเกล — สัดส่วน 0.71–0.78 (จาก 2 DOI) กับเปอร์เซ็นต์ 55–84.7
- กติกา: แถวที่ `base_property == "FF"` และค่า ≤ 1 → **× 100** (บันทึกใน `value_note` + audit)
- ช่องว่างระหว่างสองกลุ่ม (0.78 กับ 55) กว้างมาก — ไม่มีเคสก้ำกึ่ง
- ทำเฉพาะ FF เท่านั้น: `Voc` ค่า ~1.0–1.2 เป็นปกติ (หน่วย V) และ `HI` ค่าน้อย ๆ อาจ ≤ 1 ได้จริง — ห้ามเหมารวม
- ค่าแปลงแล้วอยู่ในคอลัมน์ใหม่ `value_parsed` (float) — **`value` เดิมไม่ถูกแตะ**


In [53]:
DST3 = ROOT / "processed" / "prepared"      # โฟลเดอร์ output เดียวของทั้ง pipeline

pd.options.display.max_rows = 120
tables_after_values = {name: df.copy() for name, df in tables.items()}  # snapshot ไว้เทียบใน §3.4
prep_audit = []
{name: df.shape for name, df in tables.items()}

{'devices': (95, 4),
 'layers': (760, 10),
 'materials': (1205, 16),
 'process': (1417, 9),
 'properties': (1118, 9),
 'stability': (90, 23),
 'vendors': (1551, 8)}

## 3.1 แตกชื่อ property — explore

In [54]:
n_unique = tables["properties"]["property"].nunique()
print(f"properties.property มี {n_unique} ชื่อไม่ซ้ำ — แต่ละชื่อผสมหลายมิติ (base+stat+scan+layer+carrier) เข้าด้วยกัน")
print("ตัวอย่างที่เห็นชัด:")
for n in ["Champion PCE (FS)", "HTL grain size", "Hole VTFL", "Average Voc", "Certified PCE (RS)"]:
    hits = tables["properties"][tables["properties"]["property"] == n]
    if len(hits):
        print(f"  {n!r}: {len(hits)} แถว")

properties.property มี 107 ชื่อไม่ซ้ำ — แต่ละชื่อผสมหลายมิติ (base+stat+scan+layer+carrier) เข้าด้วยกัน
ตัวอย่างที่เห็นชัด:
  'Champion PCE (FS)': 24 แถว
  'Hole VTFL': 2 แถว
  'Average Voc': 40 แถว
  'Certified PCE (RS)': 1 แถว


In [55]:
tables["properties"]["property"].value_counts().head(15)

property
Champion PCE         58
Champion Voc         52
Champion Jsc         52
Champion FF          52
Average PCE          45
Average FF           41
Average Voc          40
Average Jsc          40
VTFL                 40
Contact Angle        33
Trap Density         31
Grain size           29
Rs                   27
VBM                  27
Champion Voc (RS)    24
Name: count, dtype: int64

## 3.1 แตกชื่อ property — fix

In [56]:
def variant_key(s: str) -> str:
    return re.sub(r"[\W_]", "", s.lower())

def decompose(name: str):
    """ตัด stat -> scan -> layer -> carrier ออกจากชื่อ เหลือ base"""
    base, stat, scan, layer, carrier = name, "", "", "", ""
    m = re.match(r"^(Champion|Average|Avg|Certified|Verified)\s+(.+)$", base)
    if m:
        stat, base = m[1].lower(), m[2]
    m = re.match(r"^(.+?)\s*\((RS|FS)\)$", base)
    if m:
        base, scan = m[1], m[2]
    m = re.match(r"^(HTL|PSK|PVK)\s+(.+)$", base)
    if m:
        layer, base = m[1], m[2]
    else:
        m = re.match(r"^(.+?)\s*\((HTL|PVK|PSK)\)$", base)
        if m:
            base, layer = m[1], m[2]
    layer = {"PVK": "PSK"}.get(layer, layer)
    m = re.match(r"^(Electron|Hole)\s+(.+)$", base)
    if m:
        carrier, base = m[1].lower(), m[2]
    return base, stat, scan, layer, carrier

STAT_MERGE_PENDING = {"avg": "average", "verified": "certified"}

prop = tables["properties"]
parts = prop["property"].map(decompose)
for i, col in enumerate(["base_property", "stat", "scan", "layer", "carrier"]):
    prop[col] = parts.map(lambda t, i=i: t[i])

# ---- ชื่อที่ประกาศไว้ชัดเจน ต้องชนะการโหวตเสียงข้างมากเสมอ ----
# เหตุผล: การยุบชื่อด้วย majority vote ตัดสินจาก "ไฟล์ชุดไหนเยอะกว่า" ซึ่งพลิกได้เมื่อเพิ่มข้อมูล
#   เจอของจริงตอนเพิ่ม Raw Data ชุดที่ 2: template ใหม่เขียน "Trap Density" (D ใหญ่)
#   พอไฟล์ใหม่เยอะกว่า ชื่อ canonical เลยพลิกจาก "Trap density" -> "Trap Density"
#   ทำให้ recheck ที่อ้างชื่อเดิม match ไม่ติด และคอลัมน์ trap_density ว่างทั้ง 95 device แบบเงียบ ๆ
#   -> ชื่อที่ pipeline ปลายทางอ้างถึง ต้องตรึงไว้ที่นี่ ห้ามให้เสียงข้างมากเปลี่ยน
BASE_ALIAS = {
    # ปริมาณเดียวกัน เขียนคนละแบบ (เทียบด้วย variant_key = ตัดอักขระพิเศษ + lower)
    "trapdensity": "Trap density",
    "contactangle": "Contact angle",
    "grainsize": "Grain size",
    # ชื่อที่ variant_key ยุบให้ไม่ได้เพราะสะกดต่างกันจริง -> ต้องระบุเอง
    "eisrrec": "Rrec", "eisrs": "Rs",
    "ef": "Fermi level", "efups": "Fermi level",
    "vbmups": "VBM", "vb": "VBM",
    "electricalconductivity": "Conductivity",
    "hysteresisindex": "HI",
    "integratedjsc": "Jsc EQE",
    "bandgap": "optical band gap", "energygap": "optical band gap",
    "kpfmcpd": "Surface potential (CPD)",
    # ความหยาบผิว: เจอ 7 ชื่อสำหรับปริมาณเดียวกัน
    "afmrms": "AFM rms", "afmroughness": "AFM rms", "roughness": "AFM rms",
    "rmsroughness": "AFM rms", "roughnessrms": "AFM rms", "rms": "AFM rms", "rq": "AFM rms",
}
PREFERRED_BASE = dict(BASE_ALIAS)
groups = {}
for val, n in Counter(prop["base_property"]).items():
    groups.setdefault(variant_key(val), []).append((val, n))
base_map = {}
for k, variants in groups.items():
    canonical = PREFERRED_BASE.get(k) or max(variants, key=lambda v: (v[1], v[0]))[0]
    base_map.update({val: canonical for val, n in variants if val != canonical})
prop["base_property"] = prop["base_property"].replace(base_map)

# บังคับ alias อีกชั้นหลังโหวต -- กันเคสที่ชื่อในกลุ่มไม่ได้อยู่ใน PREFERRED_BASE โดยตรง
_alias_hit = prop["base_property"].map(lambda b: BASE_ALIAS.get(variant_key(str(b))))
_am = _alias_hit.notna() & (_alias_hit != prop["base_property"])
if _am.any():
    print("บังคับ alias (ชนะเสียงข้างมาก):",
          prop.loc[_am].groupby("base_property").size().to_dict(), "->",
          sorted(_alias_hit[_am].unique()))
    prop.loc[_am, "base_property"] = _alias_hit[_am]

print(f"ชื่อ property {prop['property'].nunique()} แบบ -> base_property {prop['base_property'].nunique()} แบบ")
print("case-merge หลังแตก:", base_map)

decomp_map = (prop.groupby(["property", "base_property", "stat", "scan", "layer", "carrier"])
                  .size().rename("n_rows").reset_index()
                  .sort_values(["base_property", "stat", "scan", "layer", "carrier"])
                  .reset_index(drop=True))
decomp_map

ชื่อ property 107 แบบ -> base_property 36 แบบ
case-merge หลังแตก: {'Contact Angle': 'Contact angle', 'Trap Density': 'Trap density', 'RMS': 'AFM rms', 'Bandgap': 'optical band gap', 'Integrated Jsc': 'Jsc EQE', 'Energy gap': 'optical band gap', 'Roughness': 'AFM rms', 'RMS roughness': 'AFM rms', 'AFM Roughness': 'AFM rms', 'Roughness RMS': 'AFM rms', 'Rq': 'AFM rms', 'Electrical conductivity': 'Conductivity', 'Hysteresis Index': 'HI', 'EIS Rs': 'Rs', 'EIS Rrec': 'Rrec', 'KPFM CPD': 'Surface potential (CPD)', 'VBM (UPS)': 'VBM', 'EF (UPS)': 'Fermi level', 'EF': 'Fermi level', 'VB': 'VBM'}


,property,base_property,stat,scan,layer,carrier,n_rows
0,AFM Roughness,AFM rms,,,,,2
1,AFM rms,AFM rms,,,,,9
2,RMS,AFM rms,,,,,2
3,RMS roughness,AFM rms,,,,,4
4,Roughness,AFM rms,,,,,4
5,Roughness RMS,AFM rms,,,,,4
6,Rq,AFM rms,,,,,2
7,AFM rms (HTL),AFM rms,,,HTL,,1
8,AFM rms (PVK),AFM rms,,,PSK,,1
9,Carrier concentration,Carrier concentration,,,,,2


In [57]:
_ba = (prop.groupby("property")
         .agg(n_rows=("property", "size"),
              base_property=("base_property", "first"),
              stat=("stat", "first"), scan=("scan", "first"),
              layer=("layer", "first"), carrier=("carrier", "first"))
         .reset_index().rename(columns={"property": "before"}))
_ba["after"] = (_ba["base_property"]
                + _ba[["stat", "scan", "layer", "carrier"]]
                  .apply(lambda r: "".join(f"  +{c}={v}" for c, v in r.items() if v), axis=1))
_ba["เปลี่ยนไหม"] = (_ba["before"] != _ba["base_property"]).map({True: "แตก/รวม", False: "เหมือนเดิม"})

print(f"before -> after: {_ba['before'].nunique()} ชื่อ -> {_ba['base_property'].nunique()} base_property")
display(_ba["เปลี่ยนไหม"].value_counts().rename("n_names").to_frame())

print("เฉพาะที่เปลี่ยน:")
display(_ba[_ba["เปลี่ยนไหม"] == "แตก/รวม"]
        .sort_values(["base_property", "before"])
        [["before", "after", "base_property", "stat", "scan", "layer", "carrier", "n_rows"]]
        .reset_index(drop=True))

print("base_property ที่มีหลายชื่อต้นทางมารวมกัน:")
_merged = (_ba.groupby("base_property")["before"]
             .agg(n_names="nunique", ชื่อต้นทาง=lambda s: "  |  ".join(sorted(s))))
display(_merged[_merged["n_names"] > 1].sort_values("n_names", ascending=False))

before -> after: 107 ชื่อ -> 36 base_property


,n_names
เปลี่ยนไหม,
แตก/รวม,76
เหมือนเดิม,31


เฉพาะที่เปลี่ยน:


,before,after,base_property,stat,scan,layer,carrier,n_rows
0,AFM Roughness,AFM rms,AFM rms,,,,,2
1,AFM rms (HTL),AFM rms +layer=HTL,AFM rms,,,HTL,,1
2,AFM rms (PVK),AFM rms +layer=PSK,AFM rms,,,PSK,,1
3,RMS,AFM rms,AFM rms,,,,,2
4,RMS roughness,AFM rms,AFM rms,,,,,4
5,Roughness,AFM rms,AFM rms,,,,,4
6,Roughness RMS,AFM rms,AFM rms,,,,,4
7,Rq,AFM rms,AFM rms,,,,,2
8,Electrical conductivity,Conductivity,Conductivity,,,,,3
9,Contact Angle,Contact angle,Contact angle,,,,,33


base_property ที่มีหลายชื่อต้นทางมารวมกัน:


,n_names,ชื่อต้นทาง
base_property,,
PCE,13,Average PCE | Average PCE (FS) | Average PCE (RS) | Avg PCE | Certified PCE (FS) | Certified PCE (RS) | ...
Jsc,10,Average Jsc | Average Jsc (FS) | Average Jsc (RS) | Avg Jsc | Champion Jsc | Champion Jsc (FS) | Champio...
FF,10,Average FF | Average FF (FS) | Average FF (RS) | Avg FF | Champion FF | Champion FF (FS) | Champion FF (...
Voc,10,Average Voc | Average Voc (FS) | Average Voc (RS) | Avg Voc | Champion Voc | Champion Voc (FS) | Champio...
AFM rms,9,AFM Roughness | AFM rms | AFM rms (HTL) | AFM rms (PVK) | RMS | RMS roughness | Roughness | Roughness ...
HI,5,Average HI | Avg HI | Champion HI | HI | Hysteresis Index
Rs,4,EIS Rs | Rs | Rs (FS) | Rs (RS)
optical band gap,4,Bandgap | Energy gap | HTL optical band gap | PSK optical band gap
Fermi level,3,EF | EF (UPS) | Fermi level


## 3.2 `value_parsed` + แปลงสเกล FF — explore

In [58]:
prop = tables["properties"]
ff_check = prop[prop["base_property"] == "FF"].copy()
ff_check["v"] = pd.to_numeric(ff_check["value"], errors="coerce")
print(f"FF ทั้งหมด {len(ff_check)} แถว")
print(f"  เป็นสัดส่วน (<=1, ต้องคูณ 100)  : {int((ff_check['v']<=1).sum())} แถว")
print(f"  เป็น % อยู่แล้ว (>1)            : {int((ff_check['v']>1).sum())} แถว")
ff_check.loc[ff_check["v"] <= 1, ["device_id", "property", "value"]]

FF ทั้งหมด 174 แถว
  เป็นสัดส่วน (<=1, ต้องคูณ 100)  : 9 แถว
  เป็น % อยู่แล้ว (>1)            : 165 แถว


,device_id,property,value
2,10.1002admi.202500231_Sample1,FF,0.76
14,10.1002admi.202500231_Sample2,FF,0.75
26,10.1002admi.202500231_Sample3,FF,0.78
38,10.1002admi.202500231_Sample4,FF,0.79
308,10.1021_acsaem.2c03434_Sample2,Average FF,0.75
778,10_1016_j_solener_2025_113286_Sample1,Average FF,0.69
782,10_1016_j_solener_2025_113286_Sample1,Champion FF,0.71
793,10_1016_j_solener_2025_113286_Sample2,Average FF,0.72
797,10_1016_j_solener_2025_113286_Sample2,Champion FF,0.72


## 3.2 `value_parsed` + แปลงสเกล FF — fix

In [59]:
# ค่าที่เขียนแบบ "1.01×10-3" / "1.01 x 10^-3" -- pandas อ่านไม่ออก ต้องแปลงเป็น 1.01e-3 ก่อน
#   (มากับ Raw Data ชุดที่ 2: Conductivity ของ solener.2024.112888)
_SCI = re.compile(rf"^\s*({NUM})\s*[×xX·]\s*10\s*\^?\s*([+-]?\d+)\s*$")

def _to_num(v):
    """คืน (ค่า, note) -- note ว่างถ้าเป็นตัวเลขปกติอยู่แล้ว"""
    if not isinstance(v, str):
        return pd.to_numeric(v, errors="coerce"), ""
    m = _SCI.fullmatch(v.strip())
    if m:
        x = float(m[1]) * 10 ** int(m[2])
        return x, f"สัญกรณ์ยกกำลัง: {v.strip()} -> {x:g}"
    return pd.to_numeric(v, errors="coerce"), ""

_pv = prop["value"].map(_to_num)
prop["value_parsed"] = _pv.map(lambda t: t[0])
prop["value_note"] = _pv.map(lambda t: t[1])
_n_sci = int((prop["value_note"] != "").sum())
if _n_sci:
    print(f"แปลงสัญกรณ์ยกกำลัง {_n_sci} แถว")
    display(prop.loc[prop["value_note"] != "", ["device_id", "property", "value", "value_parsed"]])

ff_mask = (prop["base_property"] == "FF") & prop["value_parsed"].notna() & (prop["value_parsed"] <= 1)
ff_ba = prop.loc[ff_mask, ["device_id", "property", "value_parsed"]].rename(
    columns={"value_parsed": "before"})
prop.loc[ff_mask, "value_parsed"] = prop.loc[ff_mask, "value_parsed"] * 100
prop.loc[ff_mask, "value_note"] = "FF scale: fraction -> % (x100)"
ff_ba["after"] = prop.loc[ff_mask, "value_parsed"].values

for _, r in ff_ba.iterrows():
    prep_audit.append({"table": "properties", "column": "value_parsed",
                       "device_id": r["device_id"], "property": r["property"],
                       "before": r["before"], "after": r["after"],
                       "note": "FF scale x100"})

n_ff = int((prop["base_property"] == "FF").sum())
print(f"FF ทั้งหมด {n_ff} แถว | แปลงสเกล {int(ff_mask.sum())} แถว")
print("DOI ที่ได้รับผล:",
      sorted(set(ff_ba["device_id"].str.replace(r"_Sample\d+$", "", regex=True))))
ff_ba.reset_index(drop=True)

แปลงสัญกรณ์ยกกำลัง 2 แถว


,device_id,property,value,value_parsed
185,10.1016_j.solener.2024.112888_Sample1,Conductivity,1.01×10-3,0.00101
202,10.1016_j.solener.2024.112888_Sample2,Conductivity,1.21×10-3,0.00121


FF ทั้งหมด 174 แถว | แปลงสเกล 9 แถว
DOI ที่ได้รับผล: ['10.1002admi.202500231', '10.1021_acsaem.2c03434', '10_1016_j_solener_2025_113286']


,device_id,property,before,after
0,10.1002admi.202500231_Sample1,FF,0.76,76.0
1,10.1002admi.202500231_Sample2,FF,0.75,75.0
2,10.1002admi.202500231_Sample3,FF,0.78,78.0
3,10.1002admi.202500231_Sample4,FF,0.79,79.0
4,10.1021_acsaem.2c03434_Sample2,Average FF,0.75,75.0
5,10_1016_j_solener_2025_113286_Sample1,Average FF,0.69,69.0
6,10_1016_j_solener_2025_113286_Sample1,Champion FF,0.71,71.0
7,10_1016_j_solener_2025_113286_Sample2,Average FF,0.72,72.0
8,10_1016_j_solener_2025_113286_Sample2,Champion FF,0.72,72.0


## 3.3 (ก่อนเขียน) ด่านตรวจรวม — ผ่านแล้วถึงเขียนดิสก์

รวมเงื่อนไขของ **ช่วง 1** มาไว้เป็นด่านเดียวก่อนเขียน `processed/prepared/` — ทำจนจบทุกหัวข้อแล้วค่อยตรวจทีเดียว
ถ้าไม่ผ่าน `assert` จะหยุด เซลล์เขียนดิสก์ถัดไปจะไม่ทำงาน

**ปรับ 1 เงื่อนไข**: ของเดิมเช็ค `shape ตารางไม่เปลี่ยน` ซึ่งใช้ที่จุดนี้ไม่ได้แล้ว เพราะช่วง 2/3 **ตั้งใจ**เพิ่มคอลัมน์
`_parsed`/`_note` → เปลี่ยนเป็นเช็ค **จำนวนแถว** เทียบข้อมูลดิบ + **คอลัมน์เดิมไม่ถูกแตะ** เทียบ snapshot ตอนจบช่วง 1

In [60]:
# ---- ด่านตรวจรวมก่อนเขียนดิสก์ (verify ของช่วง 1 ย้ายมาอยู่ตรงนี้) ----
BAD_CHARS_RE = (chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"])

leftover = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        groups = {}
        for val in tables[name][col].dropna():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):
                groups.setdefault(k, set()).add(val)
        leftover += [(name, col, sorted(v)) for v in groups.values() if len(v) > 1]

bad_chars = sum(int(tables[n][c].fillna("").astype(str).str.contains(BAD_CHARS_RE).sum())
                for n in tables for c in tables[n].columns)
rows_ok = all(len(tables[n]) == len(tables_raw[n]) for n in tables)
orig_ok = all(tables[n][c].fillna("").equals(tables_after_text[n][c].fillna(""))
              for n in tables_after_text for c in tables_after_text[n].columns)

print("ด่านตรวจก่อนเขียนดิสก์")
print("  1) ไม่มี variant สะกดต่างหลงเหลือ (ช่วง 1 / C) :", not leftover, leftover[:2] if leftover else "")
print("  2) ไม่มีอักขระ mu/minus/NBSP (ช่วง 1 / A)     :", bad_chars == 0, f"(เจอ {bad_chars})")
print("  3) จำนวนแถวเท่าข้อมูลดิบทุกตาราง               :", rows_ok)
print("  4) คอลัมน์เดิมไม่ถูกแตะหลังจบช่วง 1 (ช่วง 2/3)  :", orig_ok)
assert (not leftover) and bad_chars == 0 and rows_ok and orig_ok, "PRE-WRITE VERIFY FAILED — ไม่เขียนดิสก์"
print("ผ่านครบ — อนุญาตให้เขียน processed/prepared/ ได้")

ด่านตรวจก่อนเขียนดิสก์
  1) ไม่มี variant สะกดต่างหลงเหลือ (ช่วง 1 / C) : True 
  2) ไม่มีอักขระ mu/minus/NBSP (ช่วง 1 / A)     : True (เจอ 0)
  3) จำนวนแถวเท่าข้อมูลดิบทุกตาราง               : True
  4) คอลัมน์เดิมไม่ถูกแตะหลังจบช่วง 1 (ช่วง 2/3)  : True
ผ่านครบ — อนุญาตให้เขียน processed/prepared/ ได้


## 3.3 เขียนดิสก์ครั้งเดียว → `processed/prepared/` + audit

**จุดเดียวที่ pipeline เขียนตารางลงดิสก์** — `prepared/` คือผลรวมของทั้ง 3 ช่วง: ข้อความ canonical (ช่วง 1) + คอลัมน์ `_parsed`/`_note` (ช่วง 2) + property 5 มิติ และ FF สเกลเดียว (ช่วง 3)

- `properties` ได้คอลัมน์ใหม่ 7 คอลัมน์ (base/stat/scan/layer/carrier/value_parsed/value_note) + 3 คอลัมน์จาก I (value_after_unit_fix/unit_corrected/unit_fix_note)
- `processed/audit/property_decomposition.csv` — mapping ชื่อ → 5 มิติ ครบ 91 ชื่อ
- `processed/audit/prepare_audit.csv` — ทุกแถวที่ค่าถูกแปลง (FF ×100)

In [61]:
DST3.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_csv(DST3 / f"{name}.csv", index=False)
decomp_map.to_csv(AUDIT / "property_decomposition.csv", index=False)
pd.DataFrame(prep_audit).to_csv(AUDIT / "prepare_audit.csv", index=False)

print("บันทึกแล้ว:", DST3)
print("  property_decomposition.csv:", len(decomp_map), "ชื่อ")
print("  prepare_audit.csv:", len(prep_audit), "แถว (FF conversions)")

บันทึกแล้ว: D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\prepared
  property_decomposition.csv: 107 ชื่อ
  prepare_audit.csv: 9 แถว (FF conversions)


## 3.4 Verify จากไฟล์ที่เซฟจริงบนดิสก์ — 4 เงื่อนไข (จุดตรวจรวมศูนย์ของทั้ง pipeline)

1. **ไฟล์บนดิสก์ตรงกับตารางใน memory** ทุกคอลัมน์ทุกเซลล์ — การเขียนครั้งเดียวไม่เพี้ยน
2. **จำนวนแถวเท่ากับข้อมูลดิบ `processed/normalize/`** ทุกตาราง — ไม่มีแถวหาย/เกินตลอดช่วง 1→3
3. ทุกแถว property ถูกแตก (`base_property` ไม่ว่าง)
4. FF ใน `value_parsed` เป็น % ล้วน (30–100)


In [62]:
svn = {p.stem: pd.read_csv(p) for p in sorted(DST3.glob("*.csv"))}

mismatches = []
for n in tables:
    for c in tables[n].columns:
        mem, dsk = tables[n][c], svn[n][c]
        if pd.api.types.is_numeric_dtype(dsk):
            m = pd.to_numeric(mem, errors="coerce")
            same = bool((m.isna() == dsk.isna()).all()) and \
                   bool(((m - dsk).abs().fillna(0) < 1e-9).all())
        else:
            same = mem.fillna("").astype(str).equals(dsk.fillna("").astype(str))
        if not same:
            mismatches.append(f"{n}.{c}")

disk_ok = not mismatches
rows_ok = all(len(svn[n]) == len(tables_raw[n]) for n in svn)
base_ok = bool((svn["properties"]["base_property"].fillna("") != "").all())
ffv = svn["properties"].loc[svn["properties"]["base_property"] == "FF", "value_parsed"].dropna()
ff_ok = bool(((ffv > 30) & (ffv <= 100)).all())

print("1) ไฟล์บนดิสก์ตรงกับตารางใน memory       :", disk_ok, mismatches if mismatches else "")
print("2) จำนวนแถวเท่ากับข้อมูลดิบ processed/normalize :", rows_ok)
print("3) base_property ครบทุกแถว                 :", base_ok)
print(f"4) FF เป็น % ล้วน ({ffv.min():.1f}-{ffv.max():.1f})           :", ff_ok)
assert disk_ok and rows_ok and base_ok and ff_ok, "VERIFY FAILED"
print("ผ่านครบ 4 เงื่อนไข ✓ — processed/prepared/ พร้อมใช้")

1) ไฟล์บนดิสก์ตรงกับตารางใน memory       : True 
2) จำนวนแถวเท่ากับข้อมูลดิบ processed/normalize : True
3) base_property ครบทุกแถว                 : True
4) FF เป็น % ล้วน (49.0-86.3)           : True
ผ่านครบ 4 เงื่อนไข ✓ — processed/prepared/ พร้อมใช้
